## Импорты

In [ ]:
# !pip install pygwalker --quiet
!pip install gdown
# %pip install google
# %pip freeze > requirements.txt

In [ ]:
from pathlib import Path
import pandas as pd
# import pygwalker as pyg
import os
from google.colab import drive


from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.colab import auth
import google.auth
import datetime
import re
import matplotlib.pyplot as plt
import gdown
import seaborn as sns
import numpy as np

In [ ]:
FOLDER_ID = "1aoMxqBCuux2i9jBTjEmmopVfOYzYOgR-"
DATA_DIR = Path("drive_data")
dataframes = {}

url = f"https://drive.google.com/drive/folders/{FOLDER_ID}"
with_recreate_dir = (
    False  # Установите True если хотите пересоздать папку и перезалить данные
)

if with_recreate_dir:
    # Полностью удаляем папку и пересоздаём ее
    import shutil

    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(url, output=str(DATA_DIR), quiet=False, use_cookies=False)
elif not DATA_DIR.exists() or not any(DATA_DIR.glob("*.csv")):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(url, output=str(DATA_DIR), quiet=False, use_cookies=False)

for csv_path in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(csv_path)
    var_name = os.path.splitext(csv_path)[0]
    # Обрезаем 'drive_data/' из имени файла для переменной var_name
    cleaned_var_name = os.path.splitext(os.path.basename(csv_path))[0]
    dataframes[cleaned_var_name] = df
    print(f"Файл '{csv_path}' успешно загружен. Размер: {df.shape}")

print("Имена файлов:", list(dataframes.keys()))

### Содержание
1. Общие методы
1. Очистка таблиц
1. Построение критериев и формирование таргета для user_id + course_id
1. Отсечение данных для обучения модели по маске
1. Формирование датасета для обучения модели (агрегированные данные user_id + course_id с отсечением по маске) + (таргет, восстановленный по всем данным)

### Общие методы

In [ ]:
def del_vars(*vars):
    """
    Удаляет переменные из глобальной области видимости.
    """
    for _var in vars:
        globals().pop(_var, None)
    del _var

def cols_to_types(df, int_cols=[], datetime_cols=[], errors="raise"):
    """
    Приводит колонки в df к типу int64 и datetime
    """
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], errors=errors)

    for col in int_cols:
        df[col] = df[col].astype(str).str.replace(",", "", regex=False).astype(int, errors=errors)
    return df

In [ ]:
def id_cols_to_int_upd(df, cols):
    """
    Безопасное приведение столбцов к целочисленному типу с поддержкой NaN
    """
    for col in cols:
        df[col] = (
            pd.to_numeric(
                df[col].astype(str).str.replace(",", "", regex=False),
                errors="coerce"
            )
            .astype("Int64")
        )
    return

## Первичный анализ

In [ ]:
# Удалить столбцы, содержащие 'Unnamed', во всех датафреймах, начинающихся с df_
for name in dataframes.keys():
    cols_to_drop = [
        col for col in dataframes[name].columns if col.startswith("Unnamed")
    ]
    if cols_to_drop:
        dataframes[name] = dataframes[name].drop(columns=cols_to_drop)

# Колонки, помеченные как "ненужное поле" в tables_description.pdf
drop_map = {
    "user_answers": [
        "performance",
    ],
    "wk_users_courses_actions": [
        "sourceable_id",
    ],
    "award_badges": [
        "name",
    ],
    "users": [
        "remember_created_at",
        "current_sign_in_at",
        "grade_changed_at",
        "last_sign_in_at",
        "grade_checked",
        "xp",
        "d_wk_municipal_id",
        "d_wk_region_id",
        "d_updated_at",
        "last_explainer_seen_→_course",
    ],
}

# Удаляем безопасно (если колонка/датафрейм отсутствуют — без ошибки)
for name in dataframes.keys():
    if name in drop_map.keys():
        dataframes[name] = dataframes[name].drop(
            columns=drop_map[name], errors="ignore"
        )

## EDA

Очистка данных

### award_badges


### user_award_badges
не требуется

### users


In [ ]:
df_users = dataframes["users"].copy()
# Удаляем агентов
df_users = df_users[df_users["type"] == "User::Pupil"]
df_users.isnull().sum()

In [ ]:
df_users["id"] = pd.to_numeric(
    df_users["id"].astype(str).str.replace(r"\D", "", regex=True)
).astype("Int64")
df_users["created_at"] = pd.to_datetime(df_users["created_at"])

In [ ]:
# Признак d_wk_school_id - id школы пользователя - меняем на флаг with_school 0/1.
df_users["with_school"] = df_users["d_wk_school_id"].notna().astype(int)
df_users.drop(columns=["d_wk_school_id"], inplace=True)

In [ ]:
display(
    df_users["type"].value_counts()
)  # убеждаемся, что все агенты удалены, поле можно удалить
display(
    df_users["is_teacher"].value_counts()
)  # убеждаемся, что значения одинаковые, поле можно удалить
display(
    df_users["subscribed"].value_counts() / len(df_users)
)  # соотношение подписанных и не подписанных имеет сильный перекос, поле можно удалить
display(
    df_users["wk_gender"].value_counts() / len(df_users)
)  # слишком мало значений, поле можно удалить

df_users.drop(
    columns=["type", "is_teacher", "subscribed", "wk_gender", "updated_at"],
    inplace=True,
)

In [ ]:
def timezone_marge(tz):
    if tz is None:
        return "other"
    tz = str(tz)
    if tz == "Europe/Moscow" or tz is None:
        return tz
    elif tz.lower().startswith("europe/"):
        return "Europe"
    elif tz.lower().startswith("asia/"):
        return "Asia"
    else:
        return "other"


df_users["timezone_group"] = df_users["timezone"].apply(lambda x: timezone_marge(x))
df_users.drop(columns=["timezone"], inplace=True)
sns.histplot(df_users["timezone_group"])
plt.show()

In [ ]:
display(df_users["sign_in_count"].value_counts())  # кол-во логинов пользователя на LMS
df_users["sign_in_count"] = (
    df_users["sign_in_count"]
    .astype(str)
    .str.replace(r"[^\d]", "", regex=True)
    .astype(int)
)

display(df_users["sign_in_count"].describe())

In [ ]:
# Медиана отличается от среднего, значит есть выбросы
# Посмотрим на распределение
sns.histplot(df_users["sign_in_count"])
plt.show()

# Посмотрим на распределение логарифма
sns.histplot(np.log(df_users["sign_in_count"]))
plt.show()

In [ ]:
display(df_users["grade_id"].value_counts() / len(df_users))  # класс пользователя
df_users["grade_id"] = (
    df_users["grade_id"].astype(str).str.replace(r"[^\d]", "", regex=True).astype(int)
)

# Посмотрим на распределение
sns.histplot(df_users["grade_id"])
plt.show()

In [ ]:
# Сильный перекос в сторону класса 3010
# Поле неинформативное, удалим
df_users.drop(columns=["grade_id"], inplace=True)

In [ ]:
df_users.reset_index(drop=True, inplace=True)
df_users.head()

In [ ]:
dataframes["users"] = df_users

In [ ]:
del_vars('df_users')

#### Работа с датасетом users


1. Первичный анализ данных  
- Проведен анализ признаков пользователей: sign_in_count (количество входов), grade_id (класс), is_teacher (признак учителя), subscribed (признак подписки), timezone (часовой пояс), wk_gender (пол), d_wk_school_id/id школы.

2. Обработка и очистка данных  
- Поля sign_in_count и grade_id были приведены к числовому формату с удалением нецифровых символов.
- Выявлены аномалии в распределении sign_in_count (наличие выбросов, медиана отличается от среднего).
- Для анализа распределения sign_in_count построены гистограммы по значениям и их логарифмам.
- Для grade_id выявлен сильный перекос в сторону одного значения (3010), после чего колонка была признана неинформативной и удалена.

3. Итоговое состояние датасета  
- После первичной обработки и удаления неинформативных/проблемных столбцов в финальном датасете остались только следующие поля:
    - id: идентификатор пользователя
    - created_at: временная метка создания аккаунта
    - sign_in_count: количество входов пользователя на платформу
    - with_school: индикатор, привязан ли пользователь к школе
    - timezone_group: сгруппированный часовой пояс
- Были удалены следующие поля:
    - updated_at (удалено)
    - grade_id (низкая информативность, удалено)
    - is_teacher (нет в финальном датасете на данном этапе)
    - subscribed (нет в финальном датасете на данном этапе)
    - d_wk_school_id (удалено)
    - wk_gender (удалено)

Краткие выводы:
- В финальном датасете users для дальнейшего анализа оставлены только те поля, которые гарантированно имеются у всех пользователей и не содержат большого числа пропусков, а также наиболее удобны для последующей обработки.
- Колонки grade_id, is_teacher, subscribed, wk_gender, d_wk_school_id были удалены как неинформативные или с большим количеством пропусков/перекосом в распределении.
- Данные users приведены к компактному и очищенному виду — только ключевая идентификация, осн. временные признаки и минимальная база для feature engineering.

### user_answers

In [ ]:
# ua = dataframes["user_answers"].copy()
# _n0 = len(ua)


# def _strip_id_series(s: pd.Series) -> pd.Series:
#     return (
#         s.astype(str)
#         .str.replace(",", "", regex=False)
#         .str.strip()
#         .replace({"nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
#     )


# for col in ("user_id", "task_id", "resource_id"):
#     if col in ua.columns:
#         ua[col] = pd.to_numeric(_strip_id_series(ua[col]), errors="coerce").astype("Int64")

# if "submitted_at" in ua.columns:
#     ua["submitted_at"] = pd.to_datetime(ua["submitted_at"], errors="coerce")

# # Строки без ключевых идентификаторов отбрасываем
# ua = ua.dropna(subset=["user_id", "task_id"])
# _n1 = len(ua)
# print(f"После dropna(user_id, task_id): {_n0} -> {_n1} (удалено {_n0 - _n1})")

# # только ответы пользователей из актуального справочника users
# _before_users = len(ua)
# ua = ua[ua["user_id"].isin(dataframes["users"]["id"])]
# print(f"После фильтра по dataframes['users']['id']: {_before_users} -> {len(ua)} (отброшено {_before_users - len(ua)})")

# ua = ua.reset_index(drop=True)
# dataframes["user_answers"] = ua
# print("Итог user_answers:", dataframes["user_answers"].shape, dataframes["user_answers"].dtypes.to_string())

In [ ]:
# Преобразования inplace
ua = dataframes["user_answers"]
_n0 = len(ua)

# Приведение id
for col in ("user_id", "task_id", "resource_id"):
    if col in ua.columns:
        ua[col] = pd.to_numeric(
            ua[col].astype(str).str.replace(",", "", regex=False),
            errors="coerce"
        ).astype("Int64")

if "submitted_at" in ua.columns:
    ua["submitted_at"] = pd.to_datetime(ua["submitted_at"], errors="coerce")

# Фильтрация
ua = ua.dropna(subset=["user_id", "task_id"])
_n1 = len(ua)
print(f"После dropna(user_id, task_id): {_n0} -> {_n1} (удалено {_n0 - _n1})")

_before_users = len(ua)
ua = ua[ua["user_id"].isin(dataframes["users"]["id"])]
print(f"После фильтра по users: {_before_users} -> {len(ua)} (отброшено {_before_users - len(ua)})")

ua.reset_index(drop=True, inplace=True)
dataframes["user_answers"] = ua

# Освобождение памяти
import gc
gc.collect()

print("Итог user_answers:", dataframes["user_answers"].shape)

In [ ]:
dataframes["user_answers"].isna().sum()


In [ ]:
del_vars('ua', '_n0', '_n1', '_before_users')

### groups


In [ ]:
groups = dataframes["groups"].copy()

groups_clean = groups.copy()

# datetime
date_cols = [
    "starts_at",
    "pupils_notified_at",
    "wk_actual_started_at",
    "wk_actual_finished_at",
]
cols_to_types(groups_clean, datetime_cols=date_cols)

# удалить явно дефектные строки
bad_mask = (
    (
        groups_clean["starts_at"].notna() &
        groups_clean["wk_actual_finished_at"].notna() &
        (groups_clean["starts_at"] > groups_clean["wk_actual_finished_at"])
    )
    |
    (groups_clean["duration"] <= 0)
)

print("bad rows to drop:", bad_mask.sum())

groups_clean = groups_clean.loc[~bad_mask].copy()

print("clean shape:", groups_clean.shape)
print("full_duplicates:", groups_clean.duplicated().sum())
print("duplicate id rows:", groups_clean.duplicated(subset=["id"], keep=False).sum())

groups = groups_clean.copy()
dataframes["groups"] = groups

Таблица `groups` после удаления технического столбца `Unnamed: 0`:
- полных дублей не было;
- `id` был уникален.

Основные проблемы были не структурными, а логическими:
- обнаружены строки, где `starts_at > wk_actual_finished_at`;
- обнаружена строка с `duration <= 0`.

Всего выявлено `9` явно дефектных записей. Эти строки были удалены как логически противоречивые.

Также поля
- `starts_at`,
- `pupils_notified_at`,
- `wk_actual_started_at`,
- `wk_actual_finished_at`

были приведены к типу `datetime`.

После очистки:
- противоречий по временам не осталось;
- неположительной длительности не осталось;
- полных дублей и дублей по `id` нет.

Итоговая форма очищенной таблицы: `(13067, 12)`.

### homework_items


In [ ]:
homework_items = dataframes["homework_items"].copy()

homework_items_clean = homework_items.copy()

# Базовая структура
print("shape:", homework_items_clean.shape)
print("\ndtypes before cleaning:")
print(homework_items_clean.dtypes)

print("\nfull_duplicates:", homework_items_clean.duplicated().sum())
print("duplicate id rows:", homework_items_clean.duplicated(subset=["id"], keep=False).sum())

# Проверка пропусков
miss = (
    homework_items_clean.isna().sum()
    .to_frame("null_count")
    .assign(null_pct=lambda x: (x["null_count"] / len(homework_items_clean) * 100).round(4))
    .query("null_count > 0")
    .sort_values(["null_count", "null_pct"], ascending=False)
)

print("\nmissing values:")
display(miss if len(miss) else pd.DataFrame({"message": ["no nulls"]}))

# Проверка категориальных значений
print("\nresource_type value_counts:")
print(homework_items_clean["resource_type"].value_counts(dropna=False))

# Проверка ключевых дублей и порядка
print("\nduplicate (homework_id, resource_id) rows:")
print(homework_items_clean.duplicated(subset=["homework_id", "resource_id"], keep=False).sum())

print("\nduplicate (homework_id, position) rows:")
print(homework_items_clean.duplicated(subset=["homework_id", "position"], keep=False).sum())

print("\nposition min/max:", homework_items_clean["position"].min(), homework_items_clean["position"].max())
print("position <= 0:", (homework_items_clean["position"] <= 0).sum())

# Проверка приведения id к числам
homework_items_tmp = homework_items_clean.copy()
homework_items_tmp["homework_id_num"] = pd.to_numeric(
    homework_items_tmp["homework_id"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)
homework_items_tmp["resource_id_num"] = pd.to_numeric(
    homework_items_tmp["resource_id"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

print("\nhomework_id -> numeric NaN:", homework_items_tmp["homework_id_num"].isna().sum())
print("resource_id -> numeric NaN:", homework_items_tmp["resource_id_num"].isna().sum())

# Показываем строки с конфликтом позиций внутри одного homework_id
homework_items_bad_pos = homework_items_clean[
    homework_items_clean.duplicated(subset=["homework_id", "position"], keep=False)
].sort_values(["homework_id", "position", "resource_id"])

print("\nrows with duplicated (homework_id, position):", homework_items_bad_pos.shape[0])
display(homework_items_bad_pos)

# Финальная очистка
homework_items_clean["homework_id"] = (
    homework_items_clean["homework_id"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype("int64")
)

homework_items_clean["resource_id"] = (
    homework_items_clean["resource_id"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype("int64")
)

print("\nFinal cleaned table")
print("shape:", homework_items_clean.shape)
print("\ndtypes after cleaning:")
print(homework_items_clean.dtypes)

print("\nfull_duplicates:", homework_items_clean.duplicated().sum())
print("duplicate id rows:", homework_items_clean.duplicated(subset=["id"], keep=False).sum())
print(
    "duplicate (homework_id, resource_id) rows:",
    homework_items_clean.duplicated(subset=["homework_id", "resource_id"], keep=False).sum()
)
print(
    "duplicate (homework_id, position) rows:",
    homework_items_clean.duplicated(subset=["homework_id", "position"], keep=False).sum()
)
print("position <= 0:", (homework_items_clean["position"] <= 0).sum())

homework_items = homework_items_clean.copy()
dataframes["homework_items"] = homework_items

In [ ]:
del_vars(
    "miss",
    "homework_items_tmp",
    "homework_items_bad_pos",
    "homework_items_clean",
    "homework_items",
)

1. Удален технический столбец `Unnamed: 0`.
2. Проверены:
   - форма таблицы;
   - типы данных;
   - полные дубли;
   - уникальность `id`;
   - пропуски;
   - распределение `resource_type`;
   - дубли по `(homework_id, resource_id)`;
   - дубли по `(homework_id, position)`;
   - корректность значений `position`.
3. Поля `homework_id` и `resource_id` очищены от форматирующих запятых и приведены к типу `int64`.

Что показала проверка

- Таблица структурно чистая:
  - полных дублей нет;
  - `id` уникален;
  - пропусков нет.
- Поля `homework_id` и `resource_id` полностью приводятся к числовому типу без потерь.
- Дублей по паре `(homework_id, resource_id)` нет.
- Поле `position` всегда положительно.

Выявленная логическая аномалия

Обнаружены `2` строки, в которых внутри одного и того же `homework_id` повторяется одна и та же позиция:
- `homework_id = 1643`,
- `position = 2`.

При этом это не технические дубли:
- `id` разные,
- `resource_id` разные,
- обе строки относятся к `resource_type = CommonFile`.

Следовательно, это локальная коллизия порядка элементов внутри одного домашнего задания, а не повтор одной и той же записи.

### homeworks


In [ ]:
homeworks = dataframes["homeworks"].copy()

homeworks_clean = homeworks.copy()


# Базовая структура
print("shape:", homeworks_clean.shape)
print("\ndtypes before cleaning:")
print(homeworks_clean.dtypes)

print("\nfull_duplicates:", homeworks_clean.duplicated().sum())
print("duplicate id rows:", homeworks_clean.duplicated(subset=["id"], keep=False).sum())

# Проверка пропусков
miss = (
    homeworks_clean.isna().sum()
    .to_frame("null_count")
    .assign(null_pct=lambda x: (x["null_count"] / len(homeworks_clean) * 100).round(4))
    .query("null_count > 0")
    .sort_values(["null_count", "null_pct"], ascending=False)
)

print("\nmissing values:")
display(miss if len(miss) else pd.DataFrame({"message": ["no nulls"]}))

# Проверка категориальных значений
print("\nresource_type value_counts:")
print(homeworks_clean["resource_type"].value_counts(dropna=False))

print("\nhomework_type value_counts:")
print(homeworks_clean["homework_type"].value_counts(dropna=False).sort_index())

# 5. Проверка дублей по бизнес-ключу
print("\nduplicate (resource_type, resource_id) rows:")
print(homeworks_clean.duplicated(subset=["resource_type", "resource_id"], keep=False).sum())

homeworks_dup_resource = homeworks_clean[
    homeworks_clean.duplicated(subset=["resource_type", "resource_id"], keep=False)
].sort_values(["resource_type", "resource_id", "id"])

print("\nrows with duplicated (resource_type, resource_id):", homeworks_dup_resource.shape[0])
display(homeworks_dup_resource)

dup_profile = (
    homeworks_dup_resource
    .groupby(["resource_type", "resource_id"])
    .agg(
        n_rows=("id", "size"),
        n_id=("id", "nunique"),
        n_homework_type=("homework_type", "nunique"),
    )
    .reset_index()
    .sort_values(["n_rows", "n_homework_type"], ascending=False)
)

print("\nduplicate resource profile:")
display(dup_profile)

# Проверка приведения resource_id к числу
homeworks_tmp = homeworks_clean.copy()
homeworks_tmp["resource_id_num"] = pd.to_numeric(
    homeworks_tmp["resource_id"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

print("\nresource_id -> numeric NaN:", homeworks_tmp["resource_id_num"].isna().sum())

# Финальная очистка
homeworks_clean["resource_id"] = (
    homeworks_clean["resource_id"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype("int64")
)

print("\nFinal cleaned table")
print("shape:", homeworks_clean.shape)
print("\ndtypes after cleaning:")
print(homeworks_clean.dtypes)

print("\nfull_duplicates:", homeworks_clean.duplicated().sum())
print("duplicate id rows:", homeworks_clean.duplicated(subset=["id"], keep=False).sum())
print(
    "duplicate (resource_type, resource_id) rows:",
    homeworks_clean.duplicated(subset=["resource_type", "resource_id"], keep=False).sum()
)

homeworks = homeworks_clean.copy()
dataframes["homeworks"] = homeworks



In [ ]:
del_vars("miss", "homeworks_dup_resource", "dup_profile", "homeworks_tmp", "homeworks_clean", "homeworks")

1. Удален технический столбец `Unnamed: 0`.
2. Проверены:
   - форма таблицы;
   - типы данных;
   - полные дубли;
   - уникальность `id`;
   - пропуски;
   - распределения `resource_type` и `homework_type`;
   - повторы по паре `(resource_type, resource_id)`.
3. Поле `resource_id` очищено от форматирующих запятых и приведено к типу `int64`.

Что показала проверка

- Таблица имеет корректную структуру.
- Полных дублей нет.
- `id` уникален.
- Пропусков нет.
- `resource_id` полностью приводится к числовому типу без потерь.

Особенность таблицы

Были обнаружены строки с повтором пары `(resource_type, resource_id)`.  
 это **не технические дубли**:

- все такие случаи относятся к `resource_type = LessonMaterial`;
- для одного и того же `resource_id` существуют две разные записи;
- они имеют разные `id`;
- и различаются по `homework_type` (`2` и `3`).

Следовательно, эти строки отражают разные сущности домашних заданий, привязанные к одному и тому же материалу урока, и удалять их нельзя.

### lessons


### lesson_tasks

In [ ]:
lesson_tasks = dataframes["lesson_tasks"].copy()

lesson_tasks_clean = lesson_tasks.copy()


# Приводим идентификаторы к числовому виду
cols_to_types(lesson_tasks_clean, int_cols=["lesson_id", "task_id"])

# Базовые проверки структуры
print("shape:", lesson_tasks_clean.shape)
print("\ndtypes:")
print(lesson_tasks_clean.dtypes)

print("\nfull_duplicates:", lesson_tasks_clean.duplicated().sum())
print("duplicate id rows:", lesson_tasks_clean.duplicated(subset=["id"], keep=False).sum())
print(
    "duplicate (lesson_id, task_id) rows:",
    lesson_tasks_clean.duplicated(subset=["lesson_id", "task_id"], keep=False).sum()
)

# Проверка пропусков
miss = (
    lesson_tasks_clean.isna().sum()
    .to_frame("null_count")
    .assign(null_pct=lambda x: (x["null_count"] / len(lesson_tasks_clean) * 100).round(4))
    .query("null_count > 0")
    .sort_values(["null_count", "null_pct"], ascending=False)
)

print("\nmissing values:")
display(miss if len(miss) else pd.DataFrame({"message": ["no nulls"]}))

# Проверка position
print("\nposition min/max:", lesson_tasks_clean["position"].min(), lesson_tasks_clean["position"].max())
print("position <= 0:", (lesson_tasks_clean["position"] <= 0).sum())

# Логическая проверка порядка задач внутри уроков
dup_pos_rows = lesson_tasks_clean.duplicated(subset=["lesson_id", "position"], keep=False).sum()
print("\nduplicate (lesson_id, position) rows:", dup_pos_rows)

lesson_pos_summary = (
    lesson_tasks_clean
    .groupby("lesson_id")
    .agg(
        n_tasks=("task_id", "size"),
        pos_min=("position", "min"),
        pos_max=("position", "max"),
        pos_nunique=("position", "nunique"),
    )
    .reset_index()
)

lesson_pos_summary["has_gap_or_repeat"] = (
    lesson_pos_summary["pos_max"] - lesson_pos_summary["pos_min"] + 1
    != lesson_pos_summary["pos_nunique"]
)
lesson_pos_summary["starts_not_from_1"] = lesson_pos_summary["pos_min"] != 1

print("lessons with starts_not_from_1:", lesson_pos_summary["starts_not_from_1"].sum())
print("lessons with gap_or_repeat_in_positions:", lesson_pos_summary["has_gap_or_repeat"].sum())

# Точечная аномалия по повтору позиции внутри урока
lesson_tasks_bad_pos = lesson_tasks_clean[
    lesson_tasks_clean.duplicated(subset=["lesson_id", "position"], keep=False)
].sort_values(["lesson_id", "position", "task_id"])

print("\nrows with duplicated position inside lesson:", lesson_tasks_bad_pos.shape[0])
display(lesson_tasks_bad_pos)


# таблицу оставляем целиком, потому что аномалия локальная и требует предметного решения
print("\nFinal cleaned shape:", lesson_tasks_clean.shape)

dataframes["lesson_tasks"] = lesson_tasks_clean.copy()

Для таблицы `lesson_tasks` была выполнена следующая очистка:

1. Удален технический столбец `Unnamed: 0`.
2. Поля `lesson_id` и `task_id` очищены от форматирующих запятых и приведены к типу `int64`.
3. Проверены полные дубли, уникальность `id` и уникальность пары `(lesson_id, task_id)`.
4. Проверены пропуски и корректность поля `position`.

Результат очистки

Таблица после очистки имеет корректную структуру:
- полных дублей нет;
- `id` уникален;
- пары `(lesson_id, task_id)` уникальны;
- пропусков нет;
- `position` всегда положителен.

Логические проверки

Дополнительно была проверена согласованность порядка заданий внутри уроков:
- нумерация позиций во всех уроках начинается с `1`;
- глобальных дыр в диапазоне позиций не обнаружено.

При этом выявлена точечная аномалия:
- в одном уроке (`lesson_id = 561`) разные задания имеют одинаковые позиции (`2`, `3`, `7`).

Это не является дублем строк и не нарушает уникальность `(lesson_id, task_id)`, но означает локальную несогласованность порядка заданий внутри данного урока.

### trainings


### user_trainings

#### Маппинг trainig_id с course_id, определение типа треннинга

In [ ]:
# trainings join lessons -> получаем course_id
t = dataframes["trainings"][["id", "name", "lesson_id"]].copy()
t["training_id"] = t["id"].astype(str).str.replace(",", "").astype(int)
t["lesson_id"] = t["lesson_id"].astype(str).str.replace(",", "")

l = dataframes["lessons"][["id", "course_id"]].copy()
l["lesson_id"] = l["id"].astype(str).str.replace(",", "")
l["course_id"] = l["course_id"].astype(str).str.replace(",", "").astype(int)

trainings_x_courses = t.merge(
    l[["lesson_id", "course_id"]], left_on="lesson_id", right_on="lesson_id", how="left"
)

In [ ]:
# Классификация типа тренинга по имени (для критериев 4, 5, 6)
def classify_training(name):
    if pd.isna(name):
        return "other"
    name_lower = name.lower()
    if "промежуточная аттестация" in name_lower:
        return "pa"
    elif "текущий контроль" in name_lower:
        return "tk"
    elif "рефлексия" in name_lower:
        return "refl"
    elif "итоговая аттестация" in name_lower or "итоговое тестирование" in name_lower:
        return "ia"
    else:
        return "other"


trainings_x_courses["criteria_type"] = trainings_x_courses["name"].apply(
    classify_training
)

In [ ]:
# Итоговый маппинг тренниг-курс-тип треннинга
training_map = trainings_x_courses[["training_id", "course_id", "criteria_type"]]

print(
    f"training_map: {len(training_map)} записей, с course_id: {training_map['course_id'].notna().sum()}"
)
print(f"По типам:\n{training_map['criteria_type'].value_counts()}")

#### Приведение типов, джойн с маппингом

In [ ]:
ut = dataframes["user_trainings"].copy()

# ID Columns
# Datetime columns
cols_to_types(ut, int_cols=["training_id", "user_id"], datetime_cols=["started_at", "finished_at"])
ut = ut[ut["user_id"].isin(dataframes["users"]["id"])]

# Numeric columns
ut["solved_tasks_count"] = pd.to_numeric(ut["solved_tasks_count"], errors="coerce")
ut["earned_points"] = pd.to_numeric(ut["earned_points"], errors="coerce")
ut["submitted_answers_count"] = pd.to_numeric(
    ut["submitted_answers_count"], errors="coerce"
)
ut["mark"] = pd.to_numeric(ut["mark"], errors="coerce")
ut["attempts"] = pd.to_numeric(ut["attempts"], errors="coerce")

ut["duration_min"] = (ut["finished_at"] - ut["started_at"]).dt.total_seconds() / 60

In [ ]:
# Джойн с маппингом -> получаем course_id и criteria_type
ut = ut.merge(
    training_map,
    left_on="training_id",
    right_on="training_id",
    how="inner",
    suffixes=("", "_map"),
)

# Оставляем только записи с course_id
ut_with_course = ut[ut["course_id"].notna()].copy()

In [ ]:
print(
    f"\nuser_trainings с course_id: {len(ut_with_course)} из {len(dataframes['user_trainings'])}"
)
print(f"\nПо criteria_type:\n{ut_with_course['criteria_type'].value_counts()}")

#### Очистка данных

In [ ]:
# duration_min: Клиппинг выбросов по 99-м перцентилю (незакрытые сессии)
p97 = ut_with_course["duration_min"].quantile(0.97)
print(f"\nduration_min p97: {p97:.1f} мин")
ut_with_course["duration_min_clipped"] = ut_with_course["duration_min"].clip(upper=p97)

In [ ]:
# Очищенная таблица user_trainings
dataframes["user_trainings"] = ut_with_course

In [ ]:
del_vars('t', 'l', 'trainings_x_courses', 'training_map', 'ut', 'ut_with_course', 'agg_user_course', 'tk', 'crit_4', 'refl', 'crit_5', 'pa', 'crit_6', 'ut_features', 'uc', 'uc_joined')

### user_activity_histories
не используем

### user_lessons


### users_courses


In [ ]:
users_courses = dataframes["users_courses"].copy()
users_courses_raw = users_courses.copy()

print("Исходная форма:", users_courses_raw.shape)

# технический мусор
cols_to_drop = [col for col in ["Unnamed: 0"] if col in users_courses_raw.columns]
users_courses_stage1 = users_courses_raw.drop(columns=cols_to_drop).copy()

# ID-поля -> numeric
id_cols = ["id", "user_id", "course_id", "group_template_id"]
for col in id_cols:
    if col in users_courses_stage1.columns:
        users_courses_stage1[col] = (
            users_courses_stage1[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
        )
        users_courses_stage1[col] = pd.to_numeric(
            users_courses_stage1[col], errors="coerce"
        )

# Timestamp-поля дата + время
timestamp_cols = ["created_at", "updated_at", "wk_course_completed_at"]

for col in timestamp_cols:
    if col in users_courses_stage1.columns:
        users_courses_stage1[col] = pd.to_datetime(
            users_courses_stage1[col], errors="coerce"
        )
users_courses = users_courses[users_courses["user_id"].isin(dataframes["users"]["id"])]

# Date-поля: календарная дата
date_cols = ["access_finished_at", "wk_officially_started_at"]

for col in date_cols:
    if col in users_courses_stage1.columns:
        users_courses_stage1[col] = pd.to_datetime(
            users_courses_stage1[col], errors="coerce"
        )

# Числовые поля
numeric_cols = [
    "wk_points",
    "wk_max_points",
    "wk_max_viewable_lessons",
    "wk_max_task_count",
]

for col in numeric_cols:
    if col in users_courses_stage1.columns:
        users_courses_stage1[col] = (
            users_courses_stage1[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
        )
        users_courses_stage1[col] = pd.to_numeric(
            users_courses_stage1[col], errors="coerce"
        )

# Удаляем полные дубли
n_before = len(users_courses_stage1)
users_courses_stage1 = users_courses_stage1.drop_duplicates().copy()
n_after = len(users_courses_stage1)

print("После stage1:", users_courses_stage1.shape)
print("Удалено полных дублей:", n_before - n_after)

print("\nТипы после приведения:")
display(users_courses_stage1.dtypes.to_frame("dtype"))

print("\nПроверка: есть ли ненулевое время в date-полях")
date_time_check = {}
for col in date_cols:
    if col in users_courses_stage1.columns:
        non_midnight = (
            users_courses_stage1[col]
            .dropna()
            .dt.time.ne(pd.Timestamp("00:00:00").time())
            .sum()
        )
        date_time_check[col] = non_midnight

display(
    pd.DataFrame.from_dict(
        date_time_check, orient="index", columns=["n_non_midnight_values"]
    )
)

print("\nПропуски после приведения типов:")
display(
    users_courses_stage1.isna()
    .sum()
    .to_frame("null_count")
    .assign(
        null_pct=lambda df: (df["null_count"] / len(users_courses_stage1) * 100).round(
            4
        )
    )
    .sort_values("null_count", ascending=False)
)

In [ ]:
# КЛЮЧИ И БАЗОВАЯ СЕМАНТИЧЕСКАЯ ЧИСТОТА

uc2 = users_courses_stage1.copy()

print("Форма таблицы:", uc2.shape)

# Уникальность id
id_check = pd.DataFrame(
    [
        {
            "n_rows": len(uc2),
            "id_nunique": uc2["id"].nunique(dropna=True),
            "id_is_unique": uc2["id"].is_unique,
        }
    ]
)
display(id_check)

# Уникальность пары (user_id, course_id)
pair_nunique = uc2[["user_id", "course_id"]].drop_duplicates().shape[0]
pair_check = pd.DataFrame(
    [
        {
            "n_rows": len(uc2),
            "pair_nunique": pair_nunique,
            "pair_is_unique": pair_nunique == len(uc2),
        }
    ]
)
display(pair_check)

pair_dups = uc2[
    uc2.duplicated(subset=["user_id", "course_id"], keep=False)
].sort_values(["user_id", "course_id", "created_at"])

print("Число строк в нарушениях пары (user_id, course_id):", len(pair_dups))
display(pair_dups.head(20))

# Проверка state
state_check = uc2["state"].value_counts(dropna=False).to_frame("count")
display(state_check)

# Проверка id <-> (user_id, course_id)
id_to_pair = (
    uc2.groupby("id")
    .agg(n_user_id=("user_id", "nunique"), n_course_id=("course_id", "nunique"))
    .reset_index()
)

print("Есть ли id, соответствующие более чем одному user_id или course_id:")
display(
    id_to_pair[(id_to_pair["n_user_id"] > 1) | (id_to_pair["n_course_id"] > 1)].head(20)
)

In [ ]:
# удалим лишний id (зачем его вообще впихнули)
# users_courses_stage2 = users_courses_stage1.drop(columns=["id"]).copy()
users_courses_stage2 = users_courses_stage1.copy()

print("Форма после удаления id:", users_courses_stage2.shape)

print("Проверка уникальности пары (user_id, course_id) после удаления id:")
pair_nunique_after = (
    users_courses_stage2[["user_id", "course_id"]].drop_duplicates().shape[0]
)
print("n_rows =", len(users_courses_stage2))
print("pair_nunique =", pair_nunique_after)
print("pair_is_unique =", pair_nunique_after == len(users_courses_stage2))

display(users_courses_stage2.head())

In [ ]:
# СЕМАНТИЧЕСКАЯ ДИАГНОСТИКА users_courses


uc3 = users_courses_stage2.copy()

semantic_checks = pd.DataFrame(
    [
        {
            # базовая хронология
            "updated_before_created": (uc3["updated_at"] < uc3["created_at"]).sum(),
            # отдельно по смыслу статуса
            "inactive_access_finished_after_created": (
                (uc3["state"] == "inactive")
                & uc3["access_finished_at"].notna()
                & (uc3["access_finished_at"] > uc3["created_at"])
            ).sum(),
            "active_access_finished_before_created": (
                (uc3["state"] == "active")
                & uc3["access_finished_at"].notna()
                & (uc3["access_finished_at"] < uc3["created_at"])
            ).sum(),
            # учебная логика
            "completed_before_created": (
                uc3["wk_course_completed_at"].notna()
                & (uc3["wk_course_completed_at"] < uc3["created_at"])
            ).sum(),
            "completed_before_start": (
                uc3["wk_course_completed_at"].notna()
                & uc3["wk_officially_started_at"].notna()
                & (uc3["wk_course_completed_at"] < uc3["wk_officially_started_at"])
            ).sum(),
            "start_before_created": (
                uc3["wk_officially_started_at"].notna()
                & (uc3["wk_officially_started_at"] < uc3["created_at"])
            ).sum(),
            # баллы
            "points_gt_max_points": (
                uc3["wk_points"].notna()
                & uc3["wk_max_points"].notna()
                & (uc3["wk_points"] > uc3["wk_max_points"])
            ).sum(),
            "max_points_zero_but_points_positive": (
                uc3["wk_max_points"].fillna(0).eq(0) & uc3["wk_points"].fillna(0).gt(0)
            ).sum(),
            # completion vs state
            "inactive_but_completed": (
                (uc3["state"] == "inactive") & uc3["wk_course_completed_at"].notna()
            ).sum(),
            "active_but_completed": (
                (uc3["state"] == "active") & uc3["wk_course_completed_at"].notna()
            ).sum(),
        }
    ]
)

display(semantic_checks.T.rename(columns={0: "count"}))

# Покажем примеры по главным подозрительным сценариям
checks_examples = {
    "active_access_finished_before_created": (
        (uc3["state"] == "active")
        & uc3["access_finished_at"].notna()
        & (uc3["access_finished_at"] < uc3["created_at"])
    ),
    "completed_before_created": (
        uc3["wk_course_completed_at"].notna()
        & (uc3["wk_course_completed_at"] < uc3["created_at"])
    ),
    "completed_before_start": (
        uc3["wk_course_completed_at"].notna()
        & uc3["wk_officially_started_at"].notna()
        & (uc3["wk_course_completed_at"] < uc3["wk_officially_started_at"])
    ),
    "points_gt_max_points": (
        uc3["wk_points"].notna()
        & uc3["wk_max_points"].notna()
        & (uc3["wk_points"] > uc3["wk_max_points"])
    ),
    "max_points_zero_but_points_positive": (
        uc3["wk_max_points"].fillna(0).eq(0) & uc3["wk_points"].fillna(0).gt(0)
    ),
    "inactive_but_completed": (
        (uc3["state"] == "inactive") & uc3["wk_course_completed_at"].notna()
    ),
}

for name, mask in checks_examples.items():
    print(f"\n=== {name} ===")
    print("count =", mask.sum())
    display(uc3.loc[mask].head(20))

In [ ]:
# ЖЕСТКОЕ ПРАВИЛО ОЧИСТКИ ПО access_finished_at

uc4 = users_courses_stage2.copy()

uc4["created_date"] = uc4["created_at"].dt.normalize()
uc4["access_finished_date"] = uc4["access_finished_at"].dt.normalize()

# created_date - access_finished_date
uc4["created_minus_access_finished_days"] = (
    uc4["created_date"] - uc4["access_finished_date"]
).dt.days

# только для active, где access_finished_at раньше created_at по дате
mask_active_finished_before = (
    (uc4["state"] == "active")
    & uc4["access_finished_date"].notna()
    & (uc4["access_finished_date"] < uc4["created_date"])
)

active_finished_before = uc4.loc[mask_active_finished_before].copy()

print("Число active-строк, где access_finished_at_date < created_date:")
print(len(active_finished_before))

print("\nРаспределение по разнице в днях:")
display(
    active_finished_before["created_minus_access_finished_days"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

# допустимо: разница 1 день
uc4["flag_access_finished_before_created_1day"] = (
    (uc4["state"] == "active") & uc4["created_minus_access_finished_days"].eq(1)
).astype(int)

# аномалия: разница больше 1 дня
uc4["flag_access_finished_before_created_gt_1day"] = (
    (uc4["state"] == "active") & uc4["created_minus_access_finished_days"].gt(1)
).astype(int)

# прочие диагностические флаги
uc4["flag_inactive"] = (uc4["state"] == "inactive").astype(int)

uc4["flag_zero_max_points_with_positive_points"] = (
    uc4["wk_max_points"].fillna(0).eq(0) & uc4["wk_points"].fillna(0).gt(0)
).astype(int)

flag_summary = pd.DataFrame(
    {
        "flag": [
            "flag_access_finished_before_created_1day",
            "flag_access_finished_before_created_gt_1day",
            "flag_inactive",
            "flag_zero_max_points_with_positive_points",
        ],
        "count": [
            uc4["flag_access_finished_before_created_1day"].sum(),
            uc4["flag_access_finished_before_created_gt_1day"].sum(),
            uc4["flag_inactive"].sum(),
            uc4["flag_zero_max_points_with_positive_points"].sum(),
        ],
    }
)

display(flag_summary)

print("\nПримеры строк с разницей > 1 дня:")
display(
    uc4.loc[
        uc4["flag_access_finished_before_created_gt_1day"] == 1,
        [
            "user_id",
            "course_id",
            "state",
            "created_at",
            "access_finished_at",
            "created_minus_access_finished_days",
            "wk_points",
            "wk_max_points",
            "wk_officially_started_at",
            "wk_course_completed_at",
        ],
    ].head(30)
)

In [ ]:
# final clean
users_courses = uc4.drop(
    columns=[
        "created_date",
        "access_finished_date",
        "created_minus_access_finished_days",
        "flag_access_finished_before_created_1day",
        "flag_access_finished_before_created_gt_1day",
    ],
    errors="ignore",
).copy()

print("Форма users_courses:", users_courses.shape)

print("\nПроверка уникальности пары (user_id, course_id):")
pair_nunique = users_courses[["user_id", "course_id"]].drop_duplicates().shape[0]
print("n_rows =", len(users_courses))
print("pair_nunique =", pair_nunique)
print("pair_is_unique =", pair_nunique == len(users_courses))

print("\nСводка по финальным флагам:")
display(
    pd.DataFrame(
        {
            "flag": ["flag_inactive", "flag_zero_max_points_with_positive_points"],
            "count": [
                users_courses["flag_inactive"].sum(),
                users_courses["flag_zero_max_points_with_positive_points"].sum(),
            ],
        }
    )
)

display(users_courses.head())

In [ ]:
users_courses = users_courses.copy()
dataframes["users_courses"] = users_courses
dataframes["users_courses"].isna().sum()

In [ ]:
dataframes["users_courses"].isna().sum()

In [ ]:
del_vars(
    "users_courses",
    "users_courses_raw",
    "users_courses_stage1",
    "users_courses_stage2",
    "uc2",
    "uc3",
    "uc4",
    "cols_to_drop",
    "id_cols",
    "timestamp_cols",
    "date_cols",
    "numeric_cols",
    "date_time_check",
    "id_check",
    "pair_check",
    "pair_dups",
    "state_check",
    "id_to_pair",
    "semantic_checks",
    "checks_examples",
    "mask_active_finished_before",
    "active_finished_before",
    "flag_summary",
    "n_before",
    "n_after",
    "pair_nunique",
    "pair_nunique_after",
)

#### details `users_courses`

На этапе предобработки были выполнены следующие действия:

1. Удален технический столбец `Unnamed: 0`.
2. Удален столбец `id` как избыточный, поскольку:
   - `id` уникален,
   - пара `(user_id, course_id)` также уникальна,
   - следовательно, аналитическая сущность строки полностью определяется парой `(user_id, course_id)`.
3. Приведены типы:
   - идентификаторы — к числовому типу,
   - `created_at`, `updated_at`, `wk_course_completed_at` — к timestamp,
   - `access_finished_at`, `wk_officially_started_at` — к date-like datetime,
   - числовые учебные поля — к `float`.
4. Проверены полные дубликаты: полных дублей не обнаружено.
5. Проверена временная консистентность:
   - случаев `updated_at < created_at` не найдено;
   - случаев `wk_course_completed_at < created_at` не найдено;
   - случаев `wk_course_completed_at < wk_officially_started_at` не найдено;
   - после корректного сравнения по календарной дате не найдено ни одного случая, где для `state = active` дата `access_finished_at` была бы раньше даты `created_at`.
6. По результатам ответов заказчика строки со следующими сценариями не удалялись:
   - `wk_officially_started_at < created_at`,
   - `state = inactive`,
   - `state = active` и заполненный `wk_course_completed_at`.

Итог: из таблицы не удалялась ни одна строка по содержательным причинам.  
Финальная clean-версия строится как структурно очищенная таблица с двумя диагностическими флагами:
- `flag_inactive`,
- `flag_zero_max_points_with_positive_points`.

### wk_media_view_sessions
Данные по сессиям просмотров медиа-контента внутри LMS



In [ ]:
df_wk_media_view_sessions = dataframes["wk_media_view_sessions"].copy()
df_wk_media_view_sessions.info()

In [ ]:
# viewer_id == user_id
df_wk_media_view_sessions.rename(columns={"viewer_id": "user_id"}, inplace=True)

In [ ]:
# Приведение типов
df_wk_media_view_sessions["user_id"] = pd.to_numeric(
    df_wk_media_view_sessions["user_id"].astype(str).str.replace(r"\D", "", regex=True)
).astype("Int64")
df_wk_media_view_sessions["resource_id"] = pd.to_numeric(
    df_wk_media_view_sessions["resource_id"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
).astype("Int64")

df_wk_media_view_sessions["started_at"] = pd.to_datetime(
    df_wk_media_view_sessions["started_at"]
)
df_wk_media_view_sessions = df_wk_media_view_sessions[df_wk_media_view_sessions["user_id"].isin(dataframes["users"]["id"])]
# cols_to_types(df_wk_media_view_sessions, int_cols=["user_id", "resource_id"], datetime_cols=["started_at"])
df_wk_media_view_sessions.describe()

In [ ]:
# 852358 записей, пропусков нет
print(f"Дубликатов: {df_wk_media_view_sessions.duplicated().sum()}")

df_wk_media_view_sessions.drop_duplicates(inplace=True)

df_wk_media_view_sessions.info()
# 848800 записей, пропусков нет
print(f"Дубликатов: {df_wk_media_view_sessions.duplicated().sum()}")

In [ ]:
print("Соотношение типов ресурсов:")
display(
    df_wk_media_view_sessions["resource_type"].value_counts()
    / len(df_wk_media_view_sessions)
)

# id активности, в рамках которой было взаимодействие с задачей
# Не удаляю, вдруг пригодиться для связи с user_answers
print("Соотношение resource_id:")
display(df_wk_media_view_sessions["resource_id"].unique())

# удаляем строки, user_id которых отсутствует в датафрейме dataframes['users']
df_wk_media_view_sessions = df_wk_media_view_sessions[
    df_wk_media_view_sessions["user_id"].isin(dataframes["users"]["id"])
]

# segments_total и viewed_segments_count объединяем в один признак - глубина просмотра viewed_segments_count/segments_total
df_wk_media_view_sessions["viewed_segments_count"] = df_wk_media_view_sessions[
    "viewed_segments_count"
].astype(float)
df_wk_media_view_sessions["segments_total"] = df_wk_media_view_sessions[
    "segments_total"
].astype(float)

df_wk_media_view_sessions["depth"] = (
    df_wk_media_view_sessions["viewed_segments_count"]
    / df_wk_media_view_sessions["segments_total"]
)

# удаляем лишние колонки
# df_wk_media_view_sessions.drop(
#     columns=["viewed_segments_count", "segments_total"], inplace=True
# )

In [ ]:
df_wk_media_view_sessions["kind"].value_counts()

In [ ]:
df_wk_media_view_sessions.reset_index(drop=True, inplace=True)
print(
    f'уникальность по user_id: {df_wk_media_view_sessions.shape[0] == df_wk_media_view_sessions["user_id"].nunique()}'
)
df_wk_media_view_sessions.dtypes

In [ ]:
# Сохранение очищенной таблицы
dataframes["wk_media_view_sessions"] = df_wk_media_view_sessions

In [ ]:
del_vars("df_wk_media_view_sessions")

#### Работа с датасетом wk_media_view_sessions


1. Были обнаружены и удалены дубликаты — в датасете осталось 848800 уникальных записей.

2. Строки с user_id, которых не было в датафрейме пользователей (`df_users`), были удалены для согласованности данных.

3. Был объединён признак глубины просмотра видео: из полей `segments_total` и `viewed_segments_count` рассчитано новое поле `depth` как их отношение (`viewed_segments_count`/`segments_total`).

4. После расчёта глубины просмотра были удалены признаки `segments_total` и `viewed_segments_count` как неактуальные.

5. Произведена очистка индексов и базовая инфо-диагностика по датасету.

В итоге датасет содержит только уникальные, согласованные с пользователями записи, а также необходимый набор признаков для дальнейшего анализа за счёт удаления лишних и создания информативного поля глубины просмотра.

### wk_users_courses_actions

# Критерии и целевая переменная

## Критерии 4, 5, 6

In [ ]:
ut_with_course = dataframes["user_trainings"].copy()

# Критерий 4: Текущий контроль
tk = ut_with_course[ut_with_course["criteria_type"] == "tk"]
crit_4 = (
    tk.groupby(["user_id", "course_id"])
    .agg(
        crit_4_tk_checked=("state", lambda x: int((x == "checked").any())),
    )
    .reset_index()
)

# Критерий 5: Рефлексия
refl = ut_with_course[ut_with_course["criteria_type"] == "refl"]
crit_5 = (
    refl.groupby(["user_id", "course_id"])
    .agg(
        crit_5_refl_checked=("state", lambda x: int((x == "checked").any())),
    )
    .reset_index()
)

# Критерий 6: Промежуточная аттестация (earned_points >= 8)
pa = ut_with_course[ut_with_course["criteria_type"] == "pa"]
crit_6 = (
    pa.groupby(["user_id", "course_id"])
    .agg(
        crit_6_pa_passed=("earned_points", lambda x: int((x >= 8).any())),
    )
    .reset_index()
)

print(f"crit_4: {len(crit_4)}, crit_5: {len(crit_5)}, crit_6: {len(crit_6)}")

In [ ]:
# Удаление временных таблиц
del_vars("tk", "refl", "pa")

## Критерий 1

**Критерий 1: Посещён минимум 1 вебинар онлайн**
- Используемые при расчете поля: user_lessons.translation_visited + wk_media_view_sessions (kind=ulms_live)
- Комментарий: при валидации на эталонных данных было выявлено, что критерий 1 не влияет на итоговый "Статус", однако рассчитывается для более точного соответствия условию.

In [ ]:
# Подготовка связок (временные _<df> таблицы)
_uc = dataframes["users_courses"].copy()
_uc["uc_id"] = _uc["id"] if "id" in _uc.columns else _uc.index
_uc_map = _uc[["uc_id", "user_id", "course_id"]].drop_duplicates()

_l = dataframes["lessons"].copy()
_l["lesson_id"] = _l["id"] if "id" in _l.columns else _l.index
cols_to_types(_l, int_cols=["lesson_id", "course_id"])

_g = dataframes["groups"].copy()
_g["group_id"] = _g["id"]
cols_to_types(_g, int_cols=["group_id", "lesson_id"])

_g_with_course = _g.merge(
    _l[["lesson_id", "course_id"]],
    left_on="lesson_id", right_on="lesson_id", how="left"
)

In [ ]:
# Данные из user_lessons: translation_visited
_ul = dataframes["user_lessons"].copy()
cols_to_types(_ul, int_cols=["users_course_id", "lesson_id"])
cols_to_types(_uc_map, int_cols=["uc_id", "user_id", "course_id"])

_ul_mapped = _ul.drop(columns=["user_id"], errors="ignore").merge(_uc_map, left_on="users_course_id", right_on="uc_id", how="inner")

crit_1_from_ul = (
    _ul_mapped[_ul_mapped["translation_visited"] == True]
    .groupby(["user_id", "course_id"])
    .size()
    .reset_index(name="_cnt")
)
crit_1_from_ul["_src_ul"] = 1

In [ ]:
# Данные из wk_media_view_sessions: ключи при kind = ulms_live
_mvs = dataframes["wk_media_view_sessions"].copy()
if "viewer_id" in _mvs.columns:
    _mvs = _mvs.rename(columns={"viewer_id": "user_id"})

 # Только просмотры живых трансляций (онлайн-вебинары)
_live = _mvs[_mvs["kind"] == "ulms_live"]

# Получение course_id из _g_with_course для случая resource_type = 'Group'
_group_map = _g_with_course[["group_id", "course_id"]].drop_duplicates()
_live_group = _live[_live["resource_type"] == "Group"].merge(
    _group_map, left_on="resource_id", right_on="group_id", how="left"
)

# Получение course_id из lessons для случая resource_type = 'Lesson'
_lesson_map = _l[["lesson_id", "course_id"]].drop_duplicates()
_live_lesson = _live[_live["resource_type"] == "Lesson"].merge(
    _lesson_map, left_on="resource_id", right_on="lesson_id", how="left"
)

# Объединение результатов
_live_all = pd.concat([_live_group, _live_lesson])

# Получаем ключи, по котоорым были просмотры
crit_1_from_live = (
    _live_all[_live_all["course_id"].notna()]
    [["user_id", "course_id"]]
    .drop_duplicates()
)

# Флаг посещения онлайн-вебинара
crit_1_from_live["_src_live"] = 1

In [ ]:
# Объединение условий из обоих источников (ключи, по которым выполнилось хотя бы одно из условий)
crit_1 = (
    crit_1_from_ul[["user_id", "course_id", "_src_ul"]]
    .merge(crit_1_from_live[["user_id", "course_id", "_src_live"]],
           on=["user_id", "course_id"], how="outer")
)
crit_1["crit_1_webinar"] = 1

In [ ]:
# Сохранение результата
crit_1 = crit_1[["user_id", "course_id", "crit_1_webinar"]]
print(f"Критерий 1: {len(crit_1)} записей user_id & course_id с посещением вебинара")

In [ ]:
# Удаление временных таблиц
del_vars("_uc", "_uc_map", "_ul", "_ul_mapped", "_mvs", "_live",
         "_live_group", "_live_lesson", "_live_all", "_g", "_g_with_course",
         "_group_map", "_lesson_map", "_l",
         "crit_1_from_ul", "crit_1_from_live")

## Критерий 2

**Критерий 2: Просмотрено 80% занятий и 80% контента (720 ед)**
- Условие 1: Доля просмотренных уроков ≥ 80% (используется user_lessons)
- Условие 2: Суммарный объём контента ≥ 720 ед (используется wk_media_view_sessions)

In [ ]:
# Подготовка связок (временные _<df> таблицы)
_uc = dataframes["users_courses"].copy()
_uc["uc_id"] = _uc["id"] if "id" in _uc.columns else _uc.index
_uc_map = _uc[["uc_id", "user_id", "course_id"]].drop_duplicates()

_l = dataframes["lessons"].copy()
_l["lesson_id"] = _l["id"] if "id" in _l.columns else _l.index
cols_to_types(_l, int_cols=["lesson_id", "course_id"])

_g = dataframes["groups"].copy()
_g["group_id"] = _g["id"]
cols_to_types(_g, int_cols=["group_id", "lesson_id"])

_g_with_course = _g.merge(
    _l[["lesson_id", "course_id"]],
    left_on="lesson_id", right_on="lesson_id", how="left"
)
_g_with_course["course_id"] = _g_with_course["course_id"].astype("Int64")

In [ ]:
# Условие 1. Доля просмотренных уроков
_ul = dataframes["user_lessons"].copy()
cols_to_types(_ul, int_cols=["users_course_id", "lesson_id"])

_ul_mapped = _ul.drop(columns=["user_id"], errors="ignore").merge(_uc_map, left_on="users_course_id", right_on="uc_id", how="inner")
_ul_mapped["viewed"] = _ul_mapped["video_visited"] | _ul_mapped["translation_visited"]

# Расчет условия 1 для user_id & course_id
crit_2a = (
    _ul_mapped.groupby(["user_id", "course_id"])
    .agg(
        total_lessons=("lesson_id", "nunique"),
        viewed_lessons=("viewed", "sum"),
    )
    .reset_index()
)
crit_2a["lesson_view_pct"] = crit_2a["viewed_lessons"] / crit_2a["total_lessons"]
crit_2a["crit_2a_lessons_80"] = (crit_2a["lesson_view_pct"] >= 0.8).astype(int)

In [ ]:
# Условие 2. Суммарный объем контента
# Таблица с просмотрами медиаконтента
_mvs = dataframes["wk_media_view_sessions"].copy()
if "viewer_id" in _mvs.columns:
    _mvs = _mvs.rename(columns={"viewer_id": "user_id"})
cols_to_types(_mvs, int_cols=["user_id", "resource_id"])

# Привязка просмотров к course_id через resource_type
_group_map = _g_with_course[["group_id", "course_id"]].drop_duplicates()
_lesson_map = _l[["lesson_id", "course_id"]].drop_duplicates()

 # recource_type = "Group" -> через _g_with_course
_mvs_group = _mvs[_mvs["resource_type"] == "Group"].merge(
    _group_map, left_on="resource_id", right_on="group_id", how="inner"
)[["user_id", "course_id", "viewed_segments_count", "segments_total", "kind"]]

# recource_type = "Lesson" -> через lessons
_mvs_lesson = _mvs[_mvs["resource_type"] == "Lesson"].merge(
    _lesson_map, left_on="resource_id", right_on="lesson_id", how="inner"
)[["user_id", "course_id", "viewed_segments_count", "segments_total", "kind"]]

# Объединение
_mvs_all = pd.concat([_mvs_group, _mvs_lesson])

# Расчет суммы viewed_segments_count в рамках user_id & course_id
# (общий объём просмотренного контента на курсе)
crit_2b = (
    _mvs_all.groupby(["user_id", "course_id"])
    .agg(total_viewed_segments=("viewed_segments_count", "sum"))
    .reset_index()
)

# Расчет условия 2 для user_id & course_id
crit_2b["crit_2b_content_720"] = (crit_2b["total_viewed_segments"] >= 720).astype(int)

In [ ]:
# Объединение условий
cols_to_types(crit_2a, int_cols=["user_id", "course_id"])
cols_to_types(crit_2b, int_cols=["user_id", "course_id"])
crit_2 = crit_2a[["user_id", "course_id", "crit_2a_lessons_80"]].merge(
    crit_2b[["user_id", "course_id", "crit_2b_content_720"]],
    on=["user_id", "course_id"],
    how="left",
)
crit_2["crit_2b_content_720"] = crit_2["crit_2b_content_720"].fillna(0).astype(int)

# Расчет критерия
crit_2["crit_2_combined"] = (
    (crit_2["crit_2a_lessons_80"] == 1) & (crit_2["crit_2b_content_720"] == 1)
).astype(int)

print(f"Критерий 2: {len(crit_2)} записей")
print(f"Условий 1 (80% уроков): {crit_2['crit_2a_lessons_80'].sum()}")
print(f"Условие 2 (720 ед контента): {crit_2['crit_2b_content_720'].sum()}")
print(f"Выполнены оба условия: {crit_2['crit_2_combined'].sum()}")

In [ ]:
# Удаление временных таблиц
del_vars("_uc", "_uc_map", "_ul", "_ul_mapped", "_mvs",
         "_mvs_group", "_mvs_lesson", "_mvs_all",
         "_g", "_g_with_course", "_group_map", "_lesson_map", "_l",
         "crit_2a", "crit_2b")

## Критерий 3

In [ ]:
# для построения критерия 3 берем lesson_tasks (task_required True),
# join user_answers (solved True),
# join lessons (select course_id)

df_stats_primary = dataframes["stats__module_1"].copy()

df_lessons_primary = dataframes["lessons"].copy()
df_lessons_primary["course_id"] = df_lessons_primary["course_id"].astype(str).str.replace(",", "").astype(int)
df_lessons_primary = df_lessons_primary[(df_lessons_primary["course_id"].isin(df_stats_primary["course_id"])) & (df_lessons_primary["task_expected"] == 1) & (df_lessons_primary["lesson_number"] <= 11)]

print('lessons')
display(df_lessons_primary.shape)


df_lesson_tasks_primary = dataframes["lesson_tasks"][dataframes["lesson_tasks"]["task_required"] == True].copy()

df_lesson_tasks_primary["lesson_id"] = df_lesson_tasks_primary["lesson_id"].astype(str).str.replace(",", "").astype(int)
df_lesson_tasks_primary["task_id"] = df_lesson_tasks_primary["task_id"].astype(str).str.replace(",", "").astype(int)
df_lesson_tasks_primary["task_required"] = df_lesson_tasks_primary["task_required"].astype(int)
df_lesson_tasks_primary = df_lesson_tasks_primary[df_lesson_tasks_primary["lesson_id"].isin(df_lessons_primary["id"])]

print('lesson_tasks')
display(df_lesson_tasks_primary.shape)

### user_answers

df_user_answers_primary = dataframes["user_answers"].copy()
df_user_answers_primary["user_id"] = df_user_answers_primary["user_id"].astype(str).str.replace(",", "").astype(int)
df_user_answers_primary["task_id"] = df_user_answers_primary["task_id"].astype(str).str.replace(",", "").astype(int)
df_user_answers_primary["solved"] = df_user_answers_primary["solved"].fillna(0)
df_user_answers_primary["solved"] = df_user_answers_primary["solved"].astype(int)
df_user_answers_primary = df_user_answers_primary[df_user_answers_primary["task_id"].isin(df_lesson_tasks_primary["task_id"])]

print('user_answers')
display(df_user_answers_primary.shape)


In [ ]:
df_user_answers = df_user_answers_primary.copy()
df_lessons = df_lessons_primary.copy()
df_lesson_tasks = df_lesson_tasks_primary.copy()
df_stats = df_stats_primary.copy()

In [ ]:
# собрать агрегацию по user_id, course_id, sum(solved), sum(required)
df_courses = df_lessons.merge(df_lesson_tasks, left_on="id", right_on="lesson_id", how="inner").groupby(["course_id"]).agg({"task_required": "sum"}).reset_index()

# Собираем ответы с привязкой к уроку и курсу, чтобы перейти к расчету критерия на уровне user-course.
df_user_answers = df_user_answers[["user_id", "task_id", "solved"]]
df_user_answers = df_user_answers.merge(df_lesson_tasks, on="task_id", how="inner")
df_user_answers = df_user_answers.merge(df_lessons, left_on="lesson_id", right_on="id", how="inner")

# Нормализуем повторные ответы и поднимаем агрегацию до уровня урока.
df_user_answers = df_user_answers.groupby(["user_id", "course_id", "lesson_id", "task_id"]).agg({"solved": "max"}).reset_index()
df_user_answers = df_user_answers.groupby(["user_id", "course_id", "lesson_id"]).agg({"solved": "sum"}).reset_index()


In [ ]:
# Обогащаем ответы числом обязательных задач курса и считаем итог критерия 3 по user-course.
df_crit_3 = df_user_answers.merge(df_courses, on="course_id")
# df_user_ID_answers = df_user_answers[df_user_answers["user_id"] == 701741]
df_crit_3 = df_crit_3.groupby(["user_id", "course_id"]).agg({"task_required": "max", "solved": "sum"}).reset_index()

# Формируем бинарный признак: выполнены ли все обязательные задания курса.
df_crit_3['crit_3_solved_pct'] = df_crit_3['solved'].eq(df_crit_3['task_required']).map({True: "Да", False: "Нет"})


In [ ]:
# Сверяем рассчитанный критерий 3 с исходной статистикой на едином ключе user-course.
df_merged = df_stats.merge(df_crit_3, on=["user_id", "course_id"], how="inner")
check_crit_3 = df_stats.merge(df_crit_3, on=["user_id", "course_id"], how="inner")

display(check_crit_3[["crit_3_solved_pct", "Решены все обяз.ИЗ"]].value_counts())

In [ ]:
del_vars(
    "check_crit_3",
    "df_crit_3",
    "df_user_answers",
    "df_lesson_tasks",
    "df_lessons",
    "df_courses",
    "df_merged",
    "df_stats",
    "df_stats_primary",
    "df_lessons_primary",
    "df_lesson_tasks_primary",
    "df_user_answers_primary",
)

Так как не получается построить критерий 3, который бы совпадал с агрегированными данными, отказываемся от его использования.

## Построение целевой переменной

**Комментарий:**
- Целевая рассчитывается, как пересечение критериев 1, 2, 4, 5, 6 / 2, 4, 5, 6 / 4, 6

In [ ]:
# Копирование users_courses в качестве основы
uc_targets = dataframes["users_courses"][["user_id", "course_id"]].copy()
cols_to_types(uc_targets, int_cols=["user_id", "course_id"]);

In [ ]:
# Джойн критериев
uc_targets = uc_targets.merge(crit_1, on=["user_id", "course_id"], how="left")
uc_targets = uc_targets.merge(
    crit_2[["user_id", "course_id", "crit_2_combined"]],
    on=["user_id", "course_id"], how="left"
)
uc_targets = uc_targets.merge(crit_4, on=["user_id", "course_id"], how="left")
uc_targets = uc_targets.merge(crit_5, on=["user_id", "course_id"], how="left")
uc_targets = uc_targets.merge(crit_6, on=["user_id", "course_id"], how="left")

In [ ]:
# Заполнение NULL, замена на 0 (критерий не выполнен)
for col in ["crit_1_webinar", "crit_2_combined", "crit_4_tk_checked",
            "crit_5_refl_checked", "crit_6_pa_passed"]:
    uc_targets[col] = uc_targets[col].fillna(0).astype(int)

In [ ]:
# Расчет целевой переменной на основе 5 критериев
uc_targets["target"] = (
    (uc_targets["crit_1_webinar"] == 1)
    & (uc_targets["crit_2_combined"] == 1)
    & (uc_targets["crit_4_tk_checked"] == 1)
    & (uc_targets["crit_5_refl_checked"] == 1)
    & (uc_targets["crit_6_pa_passed"] == 1)
).astype(int)

In [ ]:
# Проверка без вебинара (4 критерия)
uc_targets["target_no_web"] = (
    (uc_targets["crit_2_combined"] == 1)
    & (uc_targets["crit_4_tk_checked"] == 1)
    & (uc_targets["crit_5_refl_checked"] == 1)
    & (uc_targets["crit_6_pa_passed"] == 1)
).astype(int)

In [ ]:
# Дополнительно: target на критериях 4 + 6
uc_targets["target_crit_4_6"] = (
    (uc_targets["crit_4_tk_checked"] == 1)
    & (uc_targets["crit_6_pa_passed"] == 1)
).astype(int)

In [ ]:
# Валидация
print(f"Target рассчитан для {len(uc_targets)} записей")
print(f"\nTarget на 5 критериях (с вебинаром): 1 (Завершил) = {uc_targets['target'].sum()}, 0 (Отчислен) = {(uc_targets['target']==0).sum()}")
print(f"\nTarget на 4 критериях (без вебинара): 1 (Завершил) = {uc_targets['target_no_web'].sum()}, 0 (Отчислен) = {(uc_targets['target_no_web']==0).sum()}")
print(f"Разница: {uc_targets['target_no_web'].sum() - uc_targets['target'].sum()} записей")
print(f"\nДополнительно: Target на 4 и 6 критериях: 1 (Завершил) = {uc_targets['target_crit_4_6'].sum()}, 0 (Отчислен) = {(uc_targets['target_crit_4_6']==0).sum()}")

In [ ]:
# Дополнительно: флаг завершения курса (курс завершён и была активность)
_uc = dataframes["users_courses"].copy()
id_cols_to_int_upd(_uc, ["user_id", "course_id"])
_uc["access_finished_dt"] = pd.to_datetime(
    _uc["access_finished_at"], format="mixed", dayfirst=True, errors="coerce"
)
_uc["course_ended"] = (
    (_uc["access_finished_dt"] < pd.Timestamp.now()) &
    (_uc["wk_points"].notna()) &
    (_uc["wk_points"] > 0)
).astype(int)

uc_targets = uc_targets.merge(
    _uc[["user_id", "course_id", "course_ended"]],
    on=["user_id", "course_id"],
    how="left"
)
uc_targets["course_ended"] = uc_targets["course_ended"].fillna(0).astype(int)

del_vars("_uc")

In [ ]:
# Сохранение результата
id_cols_to_int_upd(uc_targets, ["user_id", "course_id"])
dataframes["uc_targets"] = uc_targets

In [ ]:
# Удаление временных таблиц
del_vars("crit_1", "crit_2", "crit_4", "crit_5", "crit_6", "ut_with_course")

In [ ]:
# Общая статистика по критериям и целевой переменной

print("#" * 70)
print("Статистика по критериям")
print("#" * 70)

_t = dataframes["uc_targets"].copy()

# На всех данных
print(f"\nВсе записи (n={len(_t)})")
for col in ["crit_1_webinar", "crit_2_combined", "crit_4_tk_checked",
            "crit_5_refl_checked", "crit_6_pa_passed"]:
    cnt = (_t[col] == 1).sum()
    print(f"  {col} = 1: {cnt:>7} ({cnt/len(_t)*100:>5.1f}%)")

print(f"\ntarget (1+2+4+5+6): 1={_t['target'].sum():>6}, 0={(_t['target']==0).sum():>6}")
print(f"target_no_web (2+4+5+6): 1={_t['target_no_web'].sum():>6}, 0={(_t['target_no_web']==0).sum():>6}")
print(f"target_crit_4_6 (4+6): 1={_t['target_crit_4_6'].sum():>6}, 0={(_t['target_crit_4_6']==0).sum():>6}")

# Только завершённые курсы с активностью
_ended = _t[_t["course_ended"] == 1]
print(f"\nЗавершённые курсы с активностью (n={len(_ended)})")
for col in ["crit_1_webinar", "crit_2_combined", "crit_4_tk_checked",
            "crit_5_refl_checked", "crit_6_pa_passed"]:
    cnt = (_ended[col] == 1).sum()
    print(f"  {col} = 1: {cnt:>7} ({cnt/len(_ended)*100:>5.1f}%)")

print(f"\ntarget (1+2+4+5+6): 1={_ended['target'].sum():>6}, 0={(_ended['target']==0).sum():>6}, доля={_ended['target'].mean():.3f}")
print(f"target_no_web (2+4+5+6): 1={_ended['target_no_web'].sum():>6}, 0={(_ended['target_no_web']==0).sum():>6}, доля={_ended['target_no_web'].mean():.3f}")
print(f"target_crit_4_6 (4+6): 1={_ended['target_crit_4_6'].sum():>6}, 0={(_ended['target_crit_4_6']==0).sum():>6}, доля={_ended['target_crit_4_6'].mean():.3f}")

# Покрытие критериев по курсам
print(f"\n Покрытие по курсам (завершённые)")
total_courses = _ended["course_id"].nunique()
for col in ["crit_1_webinar", "crit_2_combined", "crit_4_tk_checked",
            "crit_5_refl_checked", "crit_6_pa_passed"]:
    courses_with = _ended[_ended[col] == 1]["course_id"].nunique()
    print(f"  {col}: {courses_with:>3} из {total_courses} курсов")

del_vars("_t", "_ended")

## Валидация на эталонной разметке

In [ ]:
# Эталон
_m1 = dataframes["stats__module_1"][["user_id", "course_id", "Статус"]].copy()
_m1["user_id"] = _m1["user_id"].astype("Int64")
_m1["course_id"] = _m1["course_id"].astype("Int64")

_m2 = dataframes["stats__module_2"][["user_id", "course_id", "Статус"]].copy()
_m2["user_id"] = _m2["user_id"].astype("Int64")
_m2["course_id"] = _m2["course_id"].astype("Int64")

_etalon = pd.concat([_m1, _m2], ignore_index=True)
_etalon["target_etalon"] = (_etalon["Статус"] == "Завершил").astype(int)

In [ ]:
# Джойн с рассчитанной целевой переменной
_check = _etalon.merge(
    dataframes["uc_targets"][["user_id", "course_id", "target", "target_no_web"]],
    on=["user_id", "course_id"],
    how="left",
)

# Берем склеившиеся записи
_check_valid = _check[_check["target"].notna()].copy()
_check_valid["target"] = _check_valid["target"].astype(int)
_check_valid["target_no_web"] = _check_valid["target_no_web"].astype(int)

print(f"Всего в эталоне: {len(_check)}")
print(f"Склеилось с рассчитанной целевой переменной: {len(_check_valid)}")
print(f"Не склеилось: {_check['target'].isna().sum()}")

In [ ]:
# Валидация результатов без Критерия 1
print("Валидация целевой переменной на эталоне stats_module_1/2")
print(pd.crosstab(
    _check_valid["target_etalon"].map({1: "Завершил (эталон)", 0: "Отчислен (эталон)"}),
    _check_valid["target_no_web"].map({1: "Завершил (расчет)", 0: "Отчислен (расчет)"}),
    margins=True
))

_tp = ((_check_valid["target_no_web"] == 1) & (_check_valid["target_etalon"] == 1)).sum()
_tn = ((_check_valid["target_no_web"] == 0) & (_check_valid["target_etalon"] == 0)).sum()
_fp = ((_check_valid["target_no_web"] == 1) & (_check_valid["target_etalon"] == 0)).sum()
_fn = ((_check_valid["target_no_web"] == 0) & (_check_valid["target_etalon"] == 1)).sum()

print(f"\nTP={_tp}, TN={_tn}, FP={_fp}, FN={_fn}")
print(f"Accuracy: {(_tp + _tn) / len(_check_valid):.4f}")

In [ ]:
# Валидация результатов с Критерием 1
print("Валидация целевой переменной на эталоне stats_module_1/2")
print(pd.crosstab(
    _check_valid["target_etalon"].map({1: "Завершил (эталон)", 0: "Отчислен (эталон)"}),
    _check_valid["target"].map({1: "Завершил (расчет)", 0: "Отчислен (расчет)"}),
    margins=True
))

_tp = ((_check_valid["target"] == 1) & (_check_valid["target_etalon"] == 1)).sum()
_tn = ((_check_valid["target"] == 0) & (_check_valid["target_etalon"] == 0)).sum()
_fp = ((_check_valid["target"] == 1) & (_check_valid["target_etalon"] == 0)).sum()
_fn = ((_check_valid["target"] == 0) & (_check_valid["target_etalon"] == 1)).sum()

print(f"\nTP={_tp}, TN={_tn}, FP={_fp}, FN={_fn}")
print(f"Accuracy: {(_tp + _tn) / len(_check_valid):.4f}")

In [ ]:
# Удаление временных таблиц
del_vars("_m1", "_m2", "_etalon", "_check", "_tp", "_tn", "_fp", "_fn", "_check_valid")

# Расчет и применение маски

## Расчет маски

In [ ]:
def clean_id(series):
    """Убирает запятые-разделители тысяч из ID полей"""
    return series.astype(str).str.replace(',', '')

In [ ]:
def build_lesson_mask(user_lessons_df, users_courses_df, lessons_df, cutoff_pct=0.5):
    """
    Строит маску первых N% уроков для каждого user_id & course_id (Ранжирование по lesson_in, порядок создания уроков).

    Returns: DataFrame с колонками [user_id, course_id, lesson_id]
    """
    # Подготовка user_lessons
    ul = user_lessons_df.copy()
    ul['users_course_id_clean'] = clean_id(ul['users_course_id'])
    ul['lesson_id_clean'] = clean_id(ul['lesson_id'])
    ul['lesson_id_num'] = pd.to_numeric(ul['lesson_id_clean'], errors='coerce')

    # Маппинг users_course_id для связки user_id & course_id
    uc = users_courses_df.copy()
    uc['uc_id_clean'] = clean_id(uc['id'])
    uc['user_id_clean'] = clean_id(uc['user_id'])
    uc['course_id_clean'] = clean_id(uc['course_id'])
    uc['access_finished_dt'] = pd.to_datetime(uc['access_finished_at'], format='mixed', dayfirst=True, errors='coerce')
    uc['course_ended'] = uc['access_finished_dt'] < pd.Timestamp.now()

    # Только завершённые курсы с активностью
    uc = uc[(uc['course_ended']) & (uc['wk_points'].notna()) & (uc['wk_points'] > 0)]
    uc_map = uc[['uc_id_clean', 'user_id_clean', 'course_id_clean']]

    # Джойн для добавления связки user_id & course_id
    ul_mapped = ul.merge(uc_map, left_on='users_course_id_clean', right_on='uc_id_clean', how='inner')

    # Ранжирование по lesson_id внутри user_id & course_id
    ul_mapped['lesson_rank'] = ul_mapped.groupby(
        ['user_id_clean', 'course_id_clean']
    )['lesson_id_num'].rank(method='dense')

    ul_mapped['total_lessons'] = ul_mapped.groupby(
        ['user_id_clean', 'course_id_clean']
    )['lesson_id_num'].transform('nunique')

    ul_mapped['lesson_pct'] = ul_mapped['lesson_rank'] / ul_mapped['total_lessons']

    # Маска: первые N%
    mask = ul_mapped[ul_mapped['lesson_pct'] <= cutoff_pct][
        ['user_id_clean', 'course_id_clean', 'lesson_id_clean']
    ].drop_duplicates()

    # Переименование
    mask = mask.rename(columns = {'user_id_clean': 'user_id',
                                  'course_id_clean': 'course_id',
                                  'lesson_id_clean': 'lesson_id'})

    return mask

In [ ]:
def build_time_mask(user_lessons_df, users_courses_df, user_activity_histories_df, cutoff_pct=0.5):
    """
    Строит маску первых N% по времени для каждого user_id & course_id (Ранжирование по времени активности).

    Returns: DataFrame с колонками [user_id, course_id, lesson_id]
    """
    # Подготовка user_lessons
    ul = user_lessons_df.copy()
    ul['ul_id_clean'] = clean_id(ul['id'])
    ul['users_course_id_clean'] = clean_id(ul['users_course_id'])
    ul['lesson_id_clean'] = clean_id(ul['lesson_id'])

    # Маппинг users_course_id для связки user_id & course_id + даты
    uc = users_courses_df.copy()
    uc['uc_id_clean'] = clean_id(uc['id'])
    uc['user_id_clean'] = clean_id(uc['user_id'])
    uc['course_id_clean'] = clean_id(uc['course_id'])
    uc['access_finished_dt'] = pd.to_datetime(uc['access_finished_at'], format='mixed', dayfirst=True, errors='coerce')
    uc['created_dt'] = pd.to_datetime(uc['created_at'], format='mixed', dayfirst=True, errors='coerce')
    uc['course_ended'] = uc['access_finished_dt'] < pd.Timestamp.now()

    # Только завершённые курсы с активностью
    uc_ended = uc[(uc['course_ended']) & (uc['wk_points'].notna()) & (uc['wk_points'] > 0)].copy()
    uc_ended['midpoint'] = uc_ended['created_dt'] + (uc_ended['access_finished_dt'] - uc_ended['created_dt']) * cutoff_pct

    uc_map = uc_ended[['uc_id_clean', 'user_id_clean', 'course_id_clean', 'midpoint']]

    # Джойн для добавления связки user_id & course_id
    ul_mapped = ul.merge(uc_map, left_on='users_course_id_clean', right_on='uc_id_clean', how='inner')

    # Даты активности из user_activity_histories
    uah = user_activity_histories_df.copy()
    uah['user_lesson_id_clean'] = clean_id(uah['user_lesson_id'])
    uah['action_dt'] = pd.to_datetime(uah['created_at'], format='mixed', errors='coerce')

    # Первая дата активности на каждом user_lesson
    first_action = uah.groupby('user_lesson_id_clean')['action_dt'].min().reset_index(name='first_action_dt')

    # Джойн даты к user_lessons
    ul_with_time = ul_mapped.merge(first_action, left_on='ul_id_clean', right_on='user_lesson_id_clean', how='left')

    # Фильтр: только уроки до midpoint
    mask = ul_with_time[ul_with_time['first_action_dt'] <= ul_with_time['midpoint']][
        ['user_id_clean', 'course_id_clean', 'lesson_id_clean']
    ].drop_duplicates()

    # Переименование
    mask = mask.rename(columns = {'user_id_clean': 'user_id',
                                  'course_id_clean': 'course_id',
                                  'lesson_id_clean': 'lesson_id'})

    return mask

In [ ]:
def mask_statistics(mask, users_courses_df, mask_name="Маска"):
    """
    Выводит статистику по маске.
    """
    # Подготовка users_courses
    uc = users_courses_df.copy()
    uc['user_id'] = clean_id(uc['user_id'])
    uc['course_id'] = clean_id(uc['course_id'])
    uc['access_finished_dt'] = pd.to_datetime(uc['access_finished_at'], format='mixed', dayfirst=True, errors='coerce')
    uc['course_ended'] = uc['access_finished_dt'] < pd.Timestamp.now()
    uc_ended = uc[(uc['course_ended']) & (uc['wk_points'].notna()) & (uc['wk_points'] > 0)]

    # Всего уроков в маске в разрезе user_id & course_id
    mask_uc = mask.groupby(['user_id', 'course_id'])['lesson_id'].nunique().reset_index(name='lessons_in_mask')

    # Пересечение с завершёнными курсами
    ended_keys = set(zip(uc_ended['user_id'], uc_ended['course_id']))
    mask_keys = set(zip(mask_uc['user_id'], mask_uc['course_id']))

    print(f"{'#'*60}")
    print(f" {mask_name}")
    print(f"{'#'*60}")
    print(f"\n--- Общая статистика ---")
    print(f"Записей user_id & course_id & lesson_id: {len(mask)}")
    print(f"Уникальных user_id & course_id: {len(mask_uc)}")
    print(f"Уникальных user_id: {mask_uc['user_id'].nunique()}")
    print(f"Уникальных course_id: {mask_uc['course_id'].nunique()}")

    print(f"\n--- Покрытие завершённых курсов ---")
    print(f"Всего завершённых курсов с активностью: {len(ended_keys)}")
    print(f"Покрыто маской: {len(mask_keys & ended_keys)} ({len(mask_keys & ended_keys)/len(ended_keys)*100:.1f}%)")

    print(f"\n--- Уроков в маске в разрезе user_id & course_id ---")
    print(mask_uc['lessons_in_mask'].describe().round(2).to_string())

In [ ]:
# Построение маски по урокам
mask_lessons = build_lesson_mask(dataframes['user_lessons'], dataframes['users_courses'], dataframes['lessons'], cutoff_pct=0.5)

# Построение маски по времени активности
mask_time = build_time_mask(dataframes['user_lessons'], dataframes['users_courses'], dataframes['user_activity_histories'], cutoff_pct=0.5)

print("\n")
mask_statistics(mask_lessons, dataframes['users_courses'], mask_name="Подход 1: 50% уроков (по lesson_id)")
print("\n")
mask_statistics(mask_time, dataframes['users_courses'], mask_name="Подход 2: 50% времени (по дате активности)")



In [ ]:
# Приведение типов
id_cols_to_int_upd(mask_lessons, ["user_id", "course_id", "lesson_id"])

# Удаление временных таблиц
del_vars("mask_time")

**Комментарии:**
- Подход 1 берёт строго 50% уроков (в среднем 6.4 урока) и покрывает 98% общего количества записей user_id & course_id (60.4к);
- Подход 2 берет записи по урокам за первую половину срока (время завершения курса - время первой активности по нему) и покрывает 94.9% общего количества записей user_id & course_id (58.5K);
- Подход 2 выделяет примерно в 2 раза больше уроков (в среднем 11.3) - вероятно потому что студенты проходят большую часть материала в первой половине курса. При этом покрывает меньше ключей user_id & course_id из-за дополнительной зависимости от наличия записей в `user_activity_histories`.

**Вывод:**
- Предлагается выбрать подход 1, так как:
    - Выше покрытие данных (по ключу user_id & course_id);
    - Не зависит от внешних таблиц (записей по активности);
    - Нет смещения по темпам прохождения и распределению уроков в рамках курса.

## Построение признаков с учетом маски

### Агрегаты user_trainings

**Комментарий:**
- Тренинги привязаны к урокам через lesson_id.

In [ ]:
_ut = dataframes["user_trainings"].copy()
id_cols_to_int_upd(_ut, ["user_id", "training_id", "course_id"])

In [ ]:
# Добавление lesson_id к user_trainings через trainings
_training_lesson = dataframes["trainings"][["id", "lesson_id"]].copy()
cols_to_types(_training_lesson, int_cols=["id"])
_training_lesson["lesson_id"] = pd.to_numeric(
    _training_lesson["lesson_id"].astype(str).str.replace(",", ""), errors="coerce"
).astype("Int64")

_ut = _ut.merge(
    _training_lesson.rename(columns={"id": "training_id"}),
    on="training_id",
    how="left"
)

# Фильтрация по маске: оставляются только тренинги из первых 50% уроков
_ut_masked = _ut.merge(
    mask_lessons,
    left_on=["user_id", "course_id", "lesson_id"],
    right_on=["user_id", "course_id", "lesson_id"],
    how="inner"
)

print(f"user_trainings до маски: {len(_ut)}")
print(f"user_trainings после маски: {len(_ut_masked)}")

In [ ]:
# Построение агрегатов в случае наличия данных после применения маски
if len(_ut_masked) > 0:
    ut_features_masked = (
        _ut_masked.groupby(["user_id", "course_id"])
        .agg(
            ut_total_cnt=("state", "count"),
            ut_checked_pct=("state", lambda x: (x == "checked").mean()),
            ut_mark_mean=("mark", "mean"),
            ut_mark_ge4_pct=("mark", lambda x: (x >= 4).mean()),
            ut_points_mean=("earned_points", "mean"),
            ut_solved_tasks_mean=("solved_tasks_count", "mean"),
            ut_duration_mean=("duration_min_clipped", "mean"),
        )
        .reset_index()
    )
    print(f"ut_features_masked: {ut_features_masked.shape}")
else:
    ut_features_masked = pd.DataFrame(columns=["user_id", "course_id"])
    print("Нет данных user_trainings в первых 50% уроков")

In [ ]:
# Удаление временных таблиц
del_vars("_ut", "_training_lesson", "_ut_masked")

### Агрегаты wk_media_view_sessions

**Комментарий:**
- Привязка к lesson_id через resource_id:
    - resource_type = "Lesson" -> resource_id = lesson_id;
    - resource_type = "Group" -> resource_id = groups.lesson_id;
- Далее привязка course_id через lessons.

In [ ]:
# Подготовка данных
_mvs = dataframes["wk_media_view_sessions"].copy()
if "viewer_id" in _mvs.columns:
    _mvs = _mvs.rename(columns={"viewer_id": "user_id"})
id_cols_to_int_upd(_mvs, ["user_id", "resource_id"])

_l = dataframes["lessons"].copy()
_l["lesson_id"] = _l["id"] if "id" in _l.columns else _l.index
id_cols_to_int_upd(_l, ["lesson_id", "course_id"])

_g = dataframes["groups"].copy()
_g["group_id"] = _g["id"]
id_cols_to_int_upd(_g, ["group_id", "lesson_id"])

_g_with_course = _g.merge(_l[["lesson_id", "course_id"]], on="lesson_id", how="left")

In [ ]:
# Привязка сессий к lesson_id

# resource_type = "Group"
_group_lesson_map = _g[["group_id", "lesson_id"]].drop_duplicates()

_mvs_group = _mvs[_mvs["resource_type"] == "Group"].merge(
    _group_lesson_map, left_on="resource_id", right_on="group_id", how="inner"
)

# resource_type = "Lesson"
_mvs_lesson = _mvs[_mvs["resource_type"] == "Lesson"].copy()
_mvs_lesson["lesson_id"] = _mvs_lesson["resource_id"]

# Объединение
_mvs_with_lesson = pd.concat([
    _mvs_group[["user_id", "lesson_id", "segments_total", "viewed_segments_count", "kind"]],
    _mvs_lesson[["user_id", "lesson_id", "segments_total", "viewed_segments_count", "kind"]],
])

# Добавление course_id через lessons
_mvs_with_course = _mvs_with_lesson.merge(
    _l[["lesson_id", "course_id"]], on="lesson_id", how="inner"
)

# Фильтрация по маске
_mvs_masked = _mvs_with_course.merge(
    mask_lessons,
    on=["user_id", "course_id", "lesson_id"],
    how="inner",
)

print(f"wk_media_view_sessions до маски: {len(_mvs_with_course)}")
print(f"wk_media_view_sessions после маски: {len(_mvs_masked)}")

In [ ]:
# Расчет агрегатов
mvs_features_masked = (
    _mvs_masked.groupby(["user_id", "course_id"])
    .agg(
        mvs_sessions_cnt=("segments_total", "count"),
        mvs_segments_total_sum=("segments_total", "sum"),
        mvs_segments_viewed_sum=("viewed_segments_count", "sum"),
        mvs_segments_viewed_mean=("viewed_segments_count", "mean"),
        mvs_depth_mean=("viewed_segments_count", lambda x: (x / _mvs_masked.loc[x.index, "segments_total"].replace(0, 1)).mean()),
        mvs_live_cnt=("kind", lambda x: (x == "ulms_live").sum()),
        mvs_vod_cnt=("kind", lambda x: (x == "ulms_vod").sum()),
        mvs_kinescope_cnt=("kind", lambda x: (x == "kinescope").sum()),
    )
    .reset_index()
)
print(f"mvs_features_masked: {mvs_features_masked.shape}")

In [ ]:
# Удаление временных таблиц
del_vars("_mvs", "_l", "_g", "_g_with_course", "_group_lesson_map",
         "_mvs_group", "_mvs_lesson", "_mvs_with_lesson", "_mvs_with_course", "_mvs_masked")

### Агрегаты user_lessons

**Комментарий:**
- Таблица уже содержит lesson_id и course_id, необходимые для применения маски.

In [ ]:
# Получение user_courses для связки с user_id
_uc = dataframes["users_courses"].copy()
_uc["uc_id"] = _uc["id"] if "id" in _uc.columns else _uc.index
id_cols_to_int_upd(_uc, ["uc_id", "user_id", "course_id"])

_uc_map = _uc[["uc_id", "user_id", "course_id"]].drop_duplicates()

# Получение user_lessons для построения агрегатов
_ul = dataframes["user_lessons"].copy()
id_cols_to_int_upd(_ul, ["users_course_id", "lesson_id"])

# Джойн для получения связки по ключу user_id & course_id & lesson_id для применения маски
_ul_mapped = _ul.drop(columns=["user_id"], errors="ignore").merge(_uc_map, left_on="users_course_id", right_on="uc_id", how="inner")

# Фильтрация по маске
_ul_masked = _ul_mapped.merge(
    mask_lessons,
    on=["user_id", "course_id", "lesson_id"],
    how="inner"
)
print(f"user_lessons до маски: {len(_ul_mapped)}")
print(f"user_lessons после маски: {len(_ul_masked)}")

In [ ]:
# Построение агрегатов
ul_features_masked = (
    _ul_masked.groupby(["user_id", "course_id"])
    .agg(
        ul_lessons_cnt=("lesson_id", "nunique"),
        ul_video_visited_pct=("video_visited", "mean"),
        ul_translation_visited_pct=("translation_visited", "mean"),
        ul_solved_pct=("solved", "mean"),
        ul_solved_tasks_sum=("solved_tasks_count", "sum"),
        ul_solved_tasks_mean=("solved_tasks_count", "mean"),
        ul_wk_points_sum=("wk_points", "sum"),
        ul_wk_points_mean=("wk_points", "mean"),
    )
    .reset_index()
)
print(f"ul_features_masked: {ul_features_masked.shape}")

In [ ]:
# Удаление временных таблиц
del_vars("_uc", "_uc_map", "_ul", "_ul_mapped", "_ul_masked")

### Агрегаты users

**Комментарий:**
- Агрегаты считаются по курсам полльзователя, заверешнным им до начала рассматриваемого.

In [ ]:
# Подготовка users_courses
_uc = dataframes["users_courses"].copy()
id_cols_to_int_upd(_uc, ["user_id", "course_id"])
_uc["created_dt"] = pd.to_datetime(
    _uc["created_at"], format="mixed", dayfirst=True, errors="coerce"
)
_uc["access_finished_dt"] = pd.to_datetime(
    _uc["access_finished_at"], format="mixed", dayfirst=True, errors="coerce"
)
_uc["wk_max_points_num"] = pd.to_numeric(
    _uc["wk_max_points"].astype(str).str.replace(",", ""), errors="coerce"
)
_uc["progress"] = _uc["wk_points"] / _uc["wk_max_points_num"].replace(0, pd.NA)

In [ ]:
# Общая таблица по ключам user_id & course_id с добавлением дней с момента регистрации
_users = dataframes["users"].copy()
id_cols_to_int_upd(_users, ["id"])
_users["reg_dt"] = pd.to_datetime(
    _users["created_at"], format="mixed", dayfirst=True, errors="coerce"
)

# Таблица с агрегатами
user_features = _uc[["user_id", "course_id", "created_dt"]].merge(
    _users[["id", "reg_dt"]].rename(columns={"id": "user_id"}),
    on="user_id", how="left"
)

# Расчет дней с момента регистрации
user_features["u_days_since_reg"] = (user_features["created_dt"] - user_features["reg_dt"]).dt.days
user_features = user_features.drop(columns=["created_dt", "reg_dt"])

In [ ]:
# Дополнительно: бейджи
_badges = dataframes["user_award_badges"].copy()
id_cols_to_int_upd(_badges, ["user_id", "award_badge_id"])
badges_per_user = _badges.groupby("user_id").agg(
    u_badges_cnt=("award_badge_id", "count"),
    u_badge_max_id=("award_badge_id", "max"),
).reset_index()

# Добавление к таблице с агрегатами
user_features = user_features.merge(badges_per_user, on="user_id", how="left")
user_features["u_badges_cnt"] = user_features["u_badges_cnt"].fillna(0).astype(int)
user_features["u_badge_max_id"] = user_features["u_badge_max_id"].fillna(0)

In [ ]:
# Агрегаты по завершённым прошлым курсам
_current = _uc[["user_id", "course_id", "created_dt"]].rename(
    columns={"course_id": "current_course_id", "created_dt": "current_start"}
)
_past = _uc[["user_id", "course_id", "access_finished_dt", "progress", "wk_points"]].rename(
    columns={"course_id": "past_course_id", "access_finished_dt": "past_end"}
)

# Self-join: для каждого курса берутся все другие курсы того же пользователя,
# которые закончились до начала текущего
_cross = _current.merge(_past, on="user_id", how="inner")

# Фильтрация: прошлый курс закончился до начала текущего
_cross = _cross[
    (_cross["past_end"] < _cross["current_start"]) &
    (_cross["past_course_id"] != _cross["current_course_id"])
]

In [ ]:
# Построение агрегатов по прошлым курсам
prev_courses_agg = (
    _cross.groupby(["user_id", "current_course_id"])
    .agg(
        u_prev_courses_cnt=("past_course_id", "nunique"),
        u_prev_progress_mean=("progress", "mean"),
        u_prev_progress_max=("progress", "max"),
        u_prev_points_mean=("wk_points", "mean"),
    )
    .reset_index()
    .rename(columns={"current_course_id": "course_id"})
)

# Джойн к таблице с агрегатами
user_features = user_features.merge(
    prev_courses_agg, on=["user_id", "course_id"], how="left"
)

# Запонение счетчика (0 -> нет прошлых курсов)
user_features["u_prev_courses_cnt"] = user_features["u_prev_courses_cnt"].fillna(0).astype(int)

In [ ]:
# Вывод результата
print(f"user_features: {user_features.shape}")
print(f"\nОписание:")
print(user_features.describe().round(2))
print(f"\nПропуски:")
print(user_features.isna().sum())

In [ ]:
# Удаление временных таблиц
del_vars("_uc", "_current", "_past", "_cross", "_users", "_badges",
         "badges_per_user", "prev_courses_agg")

### Агрегаты courses

**Комментарий:**
- Агрегаты строятся на уровне course_id и не требуют привязки к маске.

In [ ]:
# Подготовка таблиц
_l = dataframes["lessons"].copy()
id_cols_to_int_upd(_l, ["id", "course_id"])
_l = _l.rename(columns={"id": "lesson_id"})

_lt = dataframes["lesson_tasks"].copy()
id_cols_to_int_upd(_lt, ["lesson_id", "task_id"])

_g = dataframes["groups"].copy()
id_cols_to_int_upd(_g, ["id", "lesson_id"])
_g = _g.rename(columns={"id": "group_id"})

In [ ]:
# Агрегаты из lessons
lessons_agg = _l.groupby("course_id").agg(
    c_lessons_cnt=("lesson_id", "nunique"),
    c_max_points_per_lesson_mean=("wk_max_points", "mean"),
    c_task_count_per_lesson_mean=("wk_task_count", "mean"),
    c_video_duration_mean=("wk_video_duration", "mean"),
    c_lessons_with_tasks_pct=("task_expected", "mean"),
    c_lessons_with_conspect_pct=("conspect_expected", "mean"),
).reset_index()

In [ ]:
# Агрегаты из lesson_tasks
lt_with_course = _lt.merge(_l[["lesson_id", "course_id"]], on="lesson_id", how="inner")
tasks_agg = lt_with_course.groupby("course_id").agg(
    c_total_tasks_cnt=("task_id", "nunique"),
    c_required_tasks_cnt=("task_required", "sum"),
    c_required_tasks_pct=("task_required", "mean"),
).reset_index()

In [ ]:
# Агрегаты из groups
g_with_course = _g.merge(_l[["lesson_id", "course_id"]], on="lesson_id", how="inner")
groups_agg = g_with_course.groupby("course_id").agg(
    c_webinars_cnt=("group_id", "nunique"),
    c_webinar_duration_mean=("duration", "mean"),
).reset_index()

In [ ]:
# Сбор агрегатов в общей таблице
course_features = lessons_agg.merge(tasks_agg, on="course_id", how="left")
course_features = course_features.merge(groups_agg, on="course_id", how="left")

In [ ]:
# Заполнение в 0 для случаев, если нет вебинаров / задач
course_features["c_webinars_cnt"] = course_features["c_webinars_cnt"].fillna(0).astype(int)
course_features["c_total_tasks_cnt"] = course_features["c_total_tasks_cnt"].fillna(0).astype(int)
course_features["c_required_tasks_cnt"] = course_features["c_required_tasks_cnt"].fillna(0).astype(int)
course_features["c_webinar_duration_mean"] = course_features["c_webinar_duration_mean"].fillna(0)
course_features["c_required_tasks_pct"] = course_features["c_required_tasks_pct"].fillna(0)

In [ ]:
# Результаты:
print(f"course_features: {course_features.shape}")
print(f"Уникальных курсов: {course_features['course_id'].nunique()}")
# print(f"\nОписание:")
# print(course_features.describe().round(2))
print(f"\nПропуски:")
print(course_features.isna().sum())

In [ ]:
# Удаление временных таблиц
del_vars("_l", "_lt", "_g", "lt_with_course", "g_with_course",
         "lessons_agg", "tasks_agg", "groups_agg")

### Дополнительные агрегаты в разрезе user_id

In [ ]:
# Подготовка связок
_uc = dataframes["users_courses"].copy()
id_cols_to_int_upd(_uc, ["id", "user_id", "course_id"])
_uc_map = _uc[["id", "user_id", "course_id"]].rename(columns={"id": "uc_id"}).drop_duplicates()

In [ ]:
# Агрегаты из user_lessons (с маской)
_ul = dataframes["user_lessons"].copy()
id_cols_to_int_upd(_ul, ["users_course_id", "lesson_id"])
_ul = _ul.drop(columns=["user_id"], errors="ignore").merge(
    _uc_map, left_on="users_course_id", right_on="uc_id", how="inner"
)
_ul_masked = _ul.merge(mask_lessons, on=["user_id", "course_id", "lesson_id"], how="inner")

ul_user = _ul_masked.groupby("user_id").agg(
    # Объёмы
    u_total_lessons=("lesson_id", "nunique"),
    u_total_courses_with_lessons=("course_id", "nunique"),
    # Просмотры
    u_video_visited_pct=("video_visited", "mean"),
    u_translation_visited_pct=("translation_visited", "mean"),
    u_any_visited_pct=("video_visited", lambda x: (x | _ul_masked.loc[x.index, "translation_visited"]).mean()),
    # Решение задач
    u_lessons_solved_pct=("solved", "mean"),
    u_solved_tasks_total=("solved_tasks_count", "sum"),
    u_solved_tasks_mean=("solved_tasks_count", "mean"),
    u_solved_tasks_std=("solved_tasks_count", "std"),
    # Баллы
    u_wk_points_total=("wk_points", "sum"),
    u_wk_points_mean=("wk_points", "mean"),
    u_wk_points_std=("wk_points", "std"),
    u_wk_points_max=("wk_points", "max"),
    # Доля уроков с нулевыми баллами
    u_zero_points_pct=("wk_points", lambda x: (x == 0).mean()),
).reset_index()

print(f"ul_user: {ul_user.shape}")

In [ ]:
# Агрегаты из user_activity_histories (с маской через user_lesson_id)
_uah = dataframes["user_activity_histories"].copy()
id_cols_to_int_upd(_uah, ["user_lesson_id"])
_uah["action_dt"] = pd.to_datetime(_uah["created_at"], format="mixed", errors="coerce")

# user_lesson_id -> user_id + lesson_id + course_id
_ul_id_map = dataframes["user_lessons"][["id", "users_course_id", "lesson_id"]].copy()
id_cols_to_int_upd(_ul_id_map, ["id", "users_course_id", "lesson_id"])
_ul_id_map = _ul_id_map.merge(
    _uc_map, left_on="users_course_id", right_on="uc_id", how="inner"
)

_uah = _uah.merge(
    _ul_id_map[["id", "user_id", "course_id", "lesson_id"]].rename(columns={"id": "user_lesson_id"}),
    on="user_lesson_id", how="inner"
)

# Маска
_uah_masked = _uah.merge(mask_lessons, on=["user_id", "course_id", "lesson_id"], how="inner")

# Временные признаки
_uah_masked["hour"] = _uah_masked["action_dt"].dt.hour
_uah_masked["dayofweek"] = _uah_masked["action_dt"].dt.dayofweek
_uah_masked["is_weekend"] = (_uah_masked["dayofweek"] >= 5).astype(int)
_uah_masked["is_night"] = ((_uah_masked["hour"] >= 22) | (_uah_masked["hour"] <= 5)).astype(int)

uah_user = _uah_masked.groupby("user_id").agg(
    # Объёмы
    u_actions_total=("action", "count"),
    # По типам действий
    u_action_video_cnt=("action", lambda x: (x == "visit_video").sum()),
    u_action_conspect_cnt=("action", lambda x: (x == "show_conspect").sum()),
    u_action_translation_cnt=("action", lambda x: (x == "visit_translation").sum()),
    u_action_video_pct=("action", lambda x: (x == "visit_video").mean()),
    u_action_conspect_pct=("action", lambda x: (x == "show_conspect").mean()),
    # Временные паттерны
    u_activity_span_days=("action_dt", lambda x: (x.max() - x.min()).days if x.notna().sum() > 1 else 0),
    u_weekend_pct=("is_weekend", "mean"),
    u_night_pct=("is_night", "mean"),
    u_peak_hour=("hour", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else pd.NA),
    u_peak_dayofweek=("dayofweek", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else pd.NA),
    # Регулярность
    u_unique_days=("action_dt", lambda x: x.dt.date.nunique()),
).reset_index()

# Среднее количество действий в день
uah_user["u_actions_per_day"] = uah_user["u_actions_total"] / uah_user["u_unique_days"].replace(0, 1)

print(f"uah_user: {uah_user.shape}")

In [ ]:
# Агрегаты из wk_media_view_sessions (с маской)
_mvs = dataframes["wk_media_view_sessions"].copy()
if "viewer_id" in _mvs.columns:
    _mvs = _mvs.rename(columns={"viewer_id": "user_id"})
id_cols_to_int_upd(_mvs, ["user_id", "resource_id"])

# Привязка к lesson_id
_l = dataframes["lessons"].copy()
id_cols_to_int_upd(_l, ["id", "course_id"])
_l = _l.rename(columns={"id": "lesson_id"})

_g = dataframes["groups"].copy()
id_cols_to_int_upd(_g, ["id", "lesson_id"])
_g = _g.rename(columns={"id": "group_id"})

_mvs_group = _mvs[_mvs["resource_type"] == "Group"].merge(
    _g[["group_id", "lesson_id"]], left_on="resource_id", right_on="group_id", how="inner"
)
_mvs_lesson = _mvs[_mvs["resource_type"] == "Lesson"].copy()
_mvs_lesson["lesson_id"] = _mvs_lesson["resource_id"]

_mvs_with_lesson = pd.concat([
    _mvs_group[["user_id", "lesson_id", "segments_total", "viewed_segments_count", "kind"]],
    _mvs_lesson[["user_id", "lesson_id", "segments_total", "viewed_segments_count", "kind"]],
])
_mvs_with_lesson = _mvs_with_lesson.merge(
    _l[["lesson_id", "course_id"]], on="lesson_id", how="inner"
)

# Маска
_mvs_masked = _mvs_with_lesson.merge(
    mask_lessons, on=["user_id", "course_id", "lesson_id"], how="inner"
)

mvs_user = _mvs_masked.groupby("user_id").agg(
    u_mvs_sessions=("segments_total", "count"),
    u_mvs_viewed_total=("viewed_segments_count", "sum"),
    u_mvs_segments_total=("segments_total", "sum"),
    u_mvs_depth_mean=("viewed_segments_count", lambda x:
        (x / _mvs_masked.loc[x.index, "segments_total"].replace(0, 1)).mean()),
    u_mvs_live_pct=("kind", lambda x: (x == "ulms_live").mean()),
    u_mvs_vod_pct=("kind", lambda x: (x == "ulms_vod").mean()),
    u_mvs_kinescope_pct=("kind", lambda x: (x == "kinescope").mean()),
).reset_index()

print(f"mvs_user: {mvs_user.shape}")

In [ ]:
# Агрегаты из users_courses
_uc2 = dataframes["users_courses"].copy()
id_cols_to_int_upd(_uc2, ["user_id", "course_id"])
_uc2["created_dt"] = pd.to_datetime(_uc2["created_at"], format="mixed", dayfirst=True, errors="coerce")

uc_user = _uc2.groupby("user_id").agg(
    u_total_courses=("course_id", "nunique"),
    u_first_course_dt=("created_dt", "min"),
    u_last_course_dt=("created_dt", "max"),
).reset_index()

uc_user["u_courses_span_days"] = (uc_user["u_last_course_dt"] - uc_user["u_first_course_dt"]).dt.days
uc_user = uc_user.drop(columns=["u_first_course_dt", "u_last_course_dt"])

In [ ]:
# Агрегаты из user_award_badges
_badges = dataframes["user_award_badges"].copy()
id_cols_to_int_upd(_badges, ["user_id", "award_badge_id"])

badges_user = _badges.groupby("user_id").agg(
    u_badges_unique=("award_badge_id", "nunique")
).reset_index()

In [ ]:
# Сборка
user_level_features = uc_user.copy()
user_level_features = user_level_features.merge(badges_user, on="user_id", how="left")
user_level_features = user_level_features.merge(ul_user, on="user_id", how="left")
user_level_features = user_level_features.merge(uah_user, on="user_id", how="left")
user_level_features = user_level_features.merge(mvs_user, on="user_id", how="left")

# Заполнение: у кого нет бейджей = 0
user_level_features["u_badges_unique"] = user_level_features["u_badges_unique"].fillna(0).astype(int)

print(f"\n=== user_level_features ===")
print(f"Размер: {user_level_features.shape}")
print(f"Уникальных user_id: {user_level_features['user_id'].nunique()}")
print(f"\nПропуски:")
print(user_level_features.isna().sum())

In [ ]:
# Удаление временных таблиц
del_vars("_uc", "_uc_map", "_ul", "_ul_masked", "_uah", "_uah_masked",
         "_ul_id_map", "_mvs", "_mvs_group", "_mvs_lesson", "_mvs_with_lesson",
         "_mvs_masked", "_l", "_g", "_uc2", "_badges",
         "uc_user", "badges_user", "ul_user", "uah_user", "mvs_user")

# Построение итоговой выборки

In [ ]:
# Получение users_courses в качестве основы для выборки
merged_dataframe = dataframes["users_courses"].copy()
id_cols_to_int_upd(merged_dataframe, ["user_id", "course_id"])

In [ ]:
# Добавление целевой переменной
merged_dataframe = merged_dataframe.merge(
    dataframes["uc_targets"][["user_id", "course_id", "target", "target_no_web", "target_crit_4_6", "course_ended"]],
    on=["user_id", "course_id"],
    how="left",
)

# Users
_users = dataframes["users"].copy()
id_cols_to_int_upd(_users, ["id"])
merged_dataframe = merged_dataframe.merge(
    _users,
    left_on="user_id",
    right_on="id",
    suffixes=("_uc", "_u"),
    how="inner"
)

# Агрегаты user_trainings (с маской)
merged_dataframe = merged_dataframe.merge(
    ut_features_masked,
    on=["user_id", "course_id"],
    how="left"
)

# Агрегаты wk_media_view_sessions (с маской)
merged_dataframe = merged_dataframe.merge(
    mvs_features_masked,
    on=["user_id", "course_id"],
    how="left"
)

# Агрегаты user_lessons (с маской)
merged_dataframe = merged_dataframe.merge(
    ul_features_masked,
    on=["user_id", "course_id"],
    how="left"
)

# Агрегаты users & users_courses (по курсамЮ пройденным до начала текущего)
merged_dataframe = merged_dataframe.merge(
    user_features,
    on=["user_id", "course_id"],
    how="left"
)

# Агрегаты по course_id (характеристики курса)
merged_dataframe = merged_dataframe.merge(
    course_features,
    on="course_id",
    how="left"
)

# Агрегаты по user_id (характеристики пользователя, с маской)
merged_dataframe = merged_dataframe.merge(
    user_level_features,
    on="user_id",
    how="left",
)

# Удаление невалидных столбцов
merged_dataframe = merged_dataframe.drop(columns = ['u_total_courses', 'u_courses_span_days'])

print(f"Итоговая таблица: {merged_dataframe.shape}")
print(f"Целевая переменная заполнена: {merged_dataframe['target'].notna().sum()}")
print(f"Target = 1: {(merged_dataframe['target']==1).sum()}")
print(f"Target = 0: {(merged_dataframe['target']==0).sum()}")
print(f"\nПропуски:")
print(merged_dataframe.isna().sum().to_string())

In [ ]:
print(merged_dataframe[merged_dataframe['course_ended'] == 1].isna().sum().to_string())

In [ ]:
# Удаление временных таблиц
del_vars("_users")

**Алгоритм учета маски при добавлении агрегатов:**
1. Связать записи таблицы с lesson_id:
- user_answers: task_id -> lesson_tasks.lesson_id
- user_activity_histories: user_lesson_id -> user_lessons.id -> lesson_id
- wk_users_courses_actions: users_course_id -> users_courses + lesson_id
2. Добавить course_id: lesson_id -> lessons.course_id
3. Применить маску:
```
    table_masked = table_with_lesson.merge(
        mask_lessons,
        on=["user_id", "course_id", "lesson_id"],
        how="inner"
    )
```
4. Агрегировать до user_id & course_id
5. Добавить к merged_dataframe:
```
    merged_dataframe = merged_dataframe.merge(
        new_features, on=["user_id", "course_id"], how="left"
    )
```

**Описание рассчитанных агрегатов:**
* **`user_trainings`** (с маской 50%):
- `ut_total_cnt`: количество тренингов (тестов) в первых 50% уроков курса
- `ut_checked_pct`: доля проверенных (завершённых) тренингов из общего числа
- `ut_mark_mean`: средняя оценка за тренинги (шкала 2–5)
- `ut_mark_ge4_pct`: доля тренингов с оценкой >= 4
- `ut_points_mean`: средний набранный балл за тренинг
- `ut_solved_tasks_mean`: среднее количество решённых задач за тренинг
- `ut_duration_mean`: среднее время прохождения тренинга в минутах (после клиппинга по p97)

* **`wk_media_view_sessions`** (с маской 50%):
- `mvs_sessions_cnt`: количество сессий просмотра медиа-контента
- `mvs_segments_total_sum`: суммарное количество сегментов во всех просмотренных видео
- `mvs_segments_viewed_sum`: суммарное количество просмотренных сегментов
- `mvs_segments_viewed_mean`: среднее количество просмотренных сегментов за сессию
- `mvs_depth_mean`: средняя глубина просмотра (viewed / total) — насколько полно студент смотрит видео
- `mvs_live_cnt`: количество просмотров живых трансляций (ulms_live)
- `mvs_vod_cnt`: количество просмотров записей трансляций (ulms_vod)
- `mvs_kinescope_cnt`: количество просмотров предзаписанных видео (kinescope)

* **`user_lessons`** (с маской 50%):
- `ul_lessons_cnt`: количество уроков в первых 50% курса
- `ul_video_visited_pct`: доля уроков, где студент открыл запись вебинара
- `ul_translation_visited_pct`: доля уроков, где студент посетил онлайн-трансляцию
- `ul_solved_pct`: доля уроков, где студент решил все задачи
- `ul_solved_tasks_sum`: суммарное количество решённых задач по урокам
- `ul_solved_tasks_mean`: среднее количество решённых задач на урок
- `ul_wk_points_sum`: суммарные набранные баллы по урокам
- `ul_wk_points_mean`: средний набранный балл за урок

* **`users` & `users_courses`**:
- `u_days_since_reg`: дней с регистрации студента до начала текущего курса
- `u_badges_cnt`: количество бейджей у студента
- `u_badge_max_id`: максимальный id (уровень) бейджа
- `u_prev_courses_cnt`: количество курсов, завершённых студентом до начала текущего
- `u_prev_progress_mean`: средние набранные студентом баллы по завершённым курсам
- `u_prev_progress_max`: максимальный прогресс (wk_points/wk_max_points) студента по завершённым курсам
- `u_prev_points_mean`: средний прогресс (wk_points/wk_max_points) студента по завершённым курсам

* **`courses`**:
- `c_lessons_cnt`: количество уроков в курсе
- `c_max_points_per_lesson_mean`: средний максимум баллов за урок (характеристика сложности курса)
- `c_task_count_per_lesson_mean`: среднее количество задач на урок в курсе
- `c_video_duration_mean`: средняя длительность видео к уроку в курсе
- `c_lessons_with_tasks_pct`: доля уроков курса, к которым привязаны проверочные задания
- `c_lessons_with_conspect_pct`: доля уроков курса, к которым привязан конспект
- `c_total_tasks_cnt`: общее количество задач в курсе
- `c_required_tasks_cnt`: количество обязательных задач в курсе
- `c_required_tasks_pct`: доля обязательных задач от общего числа задач в курсе
- `c_webinars_cnt`: количество вебинаров (онлайн-занятий) в курсе
- `c_webinar_duration_mean`: средняя запланированная продолжительность вебинара в минутах по курсу

* **Дополнительно:**
* **`user_award_badges`**:
- `u_badges_unique`: количество уникальных типов бейджей

* **`user_lessons`** (с маской):
- `u_total_lessons`: общее количество посещённых уроков (в первых 50%)
- `u_total_courses_with_lessons`: количество курсов, где были уроки
- `u_video_visited_pct`: доля уроков с просмотренным видео
- `u_translation_visited_pct`: доля уроков с посещённой трансляцией
- `u_any_visited_pct`: доля уроков с любым типом просмотра
- `u_lessons_solved_pct`: доля уроков, где решены все задачи
- `u_solved_tasks_total/mean/std`: решённые задачи (сумма/среднее/std)
- `u_wk_points_total/mean/std/max`: баллы (сумма/среднее/std/макс)
- `u_zero_points_pct`: доля уроков с нулевыми баллами

* **`user_activity_histories`** (с маской):
- `u_actions_total`: общее количество действий
- `u_action_video/conspect/translation_cnt`: количество по типам
- `u_action_video/conspect_pct`: доли по типам
- `u_activity_span_days`: период активности в днях
- `u_weekend_pct`: доля действий в выходные
- `u_night_pct`: доля действий ночью (22:00–05:00)
- `u_peak_hour`: самый частый час активности
- `u_peak_dayofweek`: самый частый день недели
- `u_unique_days`: количество уникальных дней с активностью
- `u_actions_per_day`: среднее действий в день

* **`wk_media_view_sessions`** (с маской):
- `u_mvs_sessions`: общее количество сессий просмотра
- `u_mvs_viewed_total`: суммарно просмотренных сегментов
- `u_mvs_segments_total`: суммарно доступных сегментов
- `u_mvs_depth_mean`: средняя глубина просмотра
- `u_mvs_live/vod/kinescope_pct`: доли по типам медиа

### Сохренение на диск

In [ ]:
MRGED_DATA_FOLDER_ID = "1pf7F__1ZCMtMGbjkGr1NsLss7vO7zkOk"

# Локально удаляем существующий файл, если он есть
if os.path.exists("merged_dataframe.csv"):
    os.remove("merged_dataframe.csv")

# Сохраняем объединённый датафрейм в новый csv
merged_dataframe.to_csv("merged_dataframe.csv", index=False)

# Авторизация в Google Drive
auth.authenticate_user()

scopes = ["https://www.googleapis.com/auth/drive"]
creds, project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive.file"]
)
service = build("drive", "v3", credentials=creds, cache_discovery=False)

# Формируем имя файла с датой и пользователем, выгружаем на диск
user_info_service = build("oauth2", "v2", credentials=creds)
user_info = user_info_service.userinfo().get().execute()
user_email = user_info.get("email", "unknown_user")
safe_user = re.sub(r"\W+", "_", user_email.split("@")[0])

file_name = f"{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}_{safe_user}_merged_dataframe.csv"

file_metadata = {"name": file_name, "parents": [MRGED_DATA_FOLDER_ID]}
media = MediaFileUpload("/content/merged_dataframe.csv", mimetype="text/csv")
file = (
    service.files().create(body=file_metadata, media_body=media, fields="id").execute()
)
print("Uploaded file id:", file["id"], "with name:", file_name)

In [ ]:
merged_dataframe.dtypes

In [ ]:
del_vars(
    "merged_dataframe",
    "len_uc",
    "csv_path",
    "scopes",
    "creds",
    "project",
    "service",
    "user_info_service",
    "user_info",
    "user_email",
    "safe_user",
    "file_name",
    "file_metadata",
    "media",
    "file",
)

| Поле                                         | Описание                                                                                                                  |
|-----------------------------------------------|---------------------------------------------------------------------------------------------------------------------------|
| level_0, index                               | Индексы после reset_index, могут быть удалены при необходимости                                                           |
| user_id                                      | Уникальный идентификатор пользователя                                                                                     |
| course_id                                    | Уникальный идентификатор курса                                                                                            |
| state                                        | Статус пользователя на курсе (active/inactive)                                                                            |
| created_at_uc                                | Дата первого появления user_id на курсе (или anal. дата регистрации на курсе)                                             |
| updated_at                                   | Дата последнего обновления                                                                                                 |
| access_finished_at                           | Дата окончания доступа                                                                                                     |
| wk_points                                    | Набранные баллы по курсу                                                                                                   |
| wk_max_points                                | Макс. возможные баллы по курсу                                                                                            |
| wk_max_viewable_lessons                      | Количество доступных уроков                                                                                               |
| wk_max_task_count                            | Количество задач в курсе                                                                                                  |
| wk_officially_started_at                     | Дата официального старта курса                                                                                            |
| wk_course_completed_at                       | Дата завершения курса                                                                                                     |
| flag_inactive                                | Флаг неактивности user_id на курсе                                                                                        |
| flag_zero_max_points_with_positive_points     | Флаг: max_points==0, но есть набранные баллы                                                                              |
| sign_in_count                                | Количество логинов пользователя                                                                                           |
| reg_date/created_at_user                     | Дата регистрации пользователя в системе                                                                                   |
| first_activity_date                          | Дата первой активности пользователя (например, earliest created_at или действие в системе)                                |
| grade_id                                     | Группа/класс пользователя (категориальная переменная)                                                                     |
| subscribed                                   | Признак подписки                                                                                                          |
| is_teacher                                   | Признак преподавателя                                                                                                     |
| timezone                                     | Таймзона пользователя                                                                                                     |
| d_wk_school_id                               | Привязка к школе (ID школы, если есть)                                                                                    |
| wk_gender                                    | Пол пользователя (1.0 - мальчик, 2.0 - девочка)                                                                           |
| sessions_count (ut_sessions_count)            | Количество пользовательских сессий                                                                                        |
| session_avg_duration (ut_duration_mean)       | Средняя продолжительность пользовательских сессий                                                                         |
| ut_span_days                                 | Протяжённость активности пользователя, дни                                                                                |
| ut_solved_tasks_mean                         | Среднее число решённых задач                                                                                              |
| ut_solved_tasks_std                          | Стандартное отклонение по решённым задачам                                                                                |
| ut_solved_tasks_median                       | Медианное число решённых задач                                                                                            |
| ut_checked_percent_mean                      | Средний % проверенных задач                                                                                               |
| ut_duration_std                              | Стандартное отклонение продолжительности сессии                                                                           |
| ut_duration_median                           | Медианная продолжительность сессии                                                                                        |
| crit_4_tk_points/mark/checked                | Статистики по критерию 4 (например контрольные работы) — набрано баллов, максимальный балл, отмечены ли задания           |
| crit_5_refl_checked                          | Признак выполнения рефлексии по критерию 5                                                                                |
| crit_6_pa_passed, crit_6_pa_max_points/mark  | Флаги и значения по прохождению (crit_6), баллы                                                                           |
| index_wk_media_view_sessions                 | Индексы сессий просмотра видео                                                                                            |
| views_count_wk_media_view_sessions           | Количество просмотров (wk_media_view_sessions_agg)                                                                        |
| depth                                        | Средняя глубина просмотра по сессии (wk_media_view_sessions)                                                              |
| resource_id                                  | ID ресурса (lesson/group), видео в сессии                                                                                 |
| resource_type                                | Тип ресурса просмотра (Lesson или Group)                                                                                  |
| started_at                                   | Время начала просмотра (wk_media_view_sessions)                                                                           |
| kind                                         | Тип медиа контента (например, kinescope, ulms_vod, ulms_live)                                                            |
| city                                         | Предполагаемый город пользователя (если данные есть)                                                                      |
| age_group                                    | Группа возраста пользователя (если рассчитана по доступным данным)                                                        |
| device_type                                  | Тип устройства пользователя                                                                                               |
| ...                                          | Другие поля, сформированные в процессе объединения (merge), включая статистики поведения, взаимодействия, признаки, флаги |